# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

# System functionality
import os
import glob 
from pathlib import Path
import json

import config

# Data Analysis
import xarray as xr
# from windspharm.standard import VectorWind
# from windspharm.tools import prep_data, recover_data, order_latdim
import xeofs as xe
# import xesmf
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec
from matplotlib import colors as mcolors
from matplotlib import ticker as mticker
from matplotlib import colormaps

# Cartopy
from cartopy import crs as ccrs
from cartopy import feature as cfeature
from cartopy import util as cutil
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter, LongitudeLocator, LatitudeLocator

# Auxiliary Functions
from auxiliary_functions.time_utils import datetime64_to_yyyymmdd, string_to_yyyymm, convert_time_to_ns, convert_ns_to_datetime, extract_years_months, datetime64_to_ns
from auxiliary_functions import xarray_utils
from auxiliary_functions.plotting_utils import tick_labeller, set_plot_mode, get_figsize
from auxiliary_functions.logging_utils import CombinedTimeFormatter
from auxiliary_functions.annual_cycle_functions import fit_annual_cycle, apply_annual_cycle, reconstruct_annual_cycle
from auxiliary_functions.calculate_velocity_potential import calculate_velocity_potential_graphcast, calculate_velocity_potential_era5

# Logging
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

In [ ]:
# start_date = "2020-12-29T00:00:00.000000000"
# end_date = "2021-02-27T00:00:00.000000000"
# logger.info(f"Start date: {start_date}")
# logger.info(f"End date: {end_date}")

# logger.info("Starting data download script")
# def surface_level_preprocess(ds: xr.Dataset) -> xr.Dataset:
#     # Subset to a specific region and keep only one variable
#     subset_data = ds.sel(
#         # level=pressure_levels,
#         time=ds.time.where(ds['time'].dt.hour.isin([0, 6, 12, 18]), drop=True)
#     )

#     target_grid = xr.Dataset(
#         {
#             "lat": (["lat"], np.arange(-90, 91, 1.0)),
#             "lon": (["lon"], np.arange(0, 360, 1.0)),
#         }
#     )
#     regridder = xe.Regridder(subset_data, target_grid, "bilinear", reuse_weights=False) 

#     return regridder(subset_data) # type: ignore

# def pressure_level_preprocess(ds: xr.Dataset) -> xr.Dataset:
#     # Subset to a specific region and keep only one variable
#     subset_data = ds.sel(
#         level=pressure_levels,
#         time=ds.time.where(ds['time'].dt.hour.isin([0, 6, 12, 18]), drop=True)
#     ).assign_coords({'level': pressure_levels.astype(np.int32)})

#     target_grid = xr.Dataset(
#         {
#             "lat": (["lat"], np.arange(-90, 91, 1.0)),
#             "lon": (["lon"], np.arange(0, 360, 1.0)),
#         }
#     )
#     regridder = xe.Regridder(subset_data, target_grid, "bilinear", reuse_weights=False) 

#     return regridder(subset_data) # type: ignore

# yyyymm_strings = pd.date_range(
#     pd.to_datetime(start_date).to_period("M").to_timestamp(),
#     pd.to_datetime(end_date).to_period("M").to_timestamp(),
#     freq="MS"
# ).strftime("%Y%m")

# logger.info(f"Dates: {np.datetime64(start_date).astype('datetime64[h]')} : {np.datetime64(end_date).astype('datetime64[h]')}")

# graphcast_data_directory = f"/glade/u/home/sressel/spencer-scratch/graphcast_input_data/{string_to_yyyymm(start_date)}_{string_to_yyyymm(end_date)}"
# if not os.path.exists(graphcast_data_directory):
#     logger.info("Creating output directory...")
#     os.makedirs(graphcast_data_directory, exist_ok=True)
# logger.info(f"Graphcast data directory: {graphcast_data_directory}")

# logger.info("Load all data into a single dataset")
# all_data = xr.open_mfdataset(f"{graphcast_data_directory}/*.nc")
# logger.info(f"    Saving data...")
# all_data.to_netcdf(f"{graphcast_data_directory}/era5_data.nc")
# logger.info("Finished")

In [ ]:
years, months = extract_years_months(start_date, end_date)

# Surface level variables
logger.info("Surface level variables")
surface_base = "/gdex/data/d633000/e5.oper.an.sfc"

surface_variables = {
    "2m_temperature": "2t",
    "mean_sea_level_pressure": "msl",
    "10m_u_component_of_wind": "10u",
    "10m_v_component_of_wind": "10v"
}

surface_variables_old_names = {
    "2m_temperature": "VAR_2T",
    "mean_sea_level_pressure": "MSL",
    "10m_u_component_of_wind": "VAR_10U",
    "10m_v_component_of_wind": "VAR_10V"
}

for variable in surface_variables.keys():
    files_list = []

    logger.info(f"  {variable}")
    for ym in yyyymm_strings:
        pattern = f"{surface_base}/{ym}/e5.oper.an.sfc.*_{surface_variables[variable]}.*.nc"
        files_list.extend(glob.glob(pattern))


# Load RMM data

In [ ]:
def load_txt_as_xarray(path):
    # Read the whitespace-delimited file
    df = pd.read_csv(
        path,
        delim_whitespace=True,
        header=None,
        names=["year", "month", "day", "hour", "var1", "var2", "var3"],
        na_values=[-99.0],
    )

    # Build datetime64 index
    df["time"] = pd.to_datetime(df[["year", "month", "day", "hour"]])

    # Set time as index
    df = df.set_index("time")

    # Drop the original date columns
    df = df.drop(columns=["year", "month", "day", "hour"])

    # Convert to xarray Dataset
    ds = xr.Dataset.from_dataframe(df)

    return ds

# Example usage:
RMM_indices = load_txt_as_xarray("RMM.txt")
RMM1 = RMM_indices["var1"]
RMM2 = RMM_indices["var2"]
amplitude = RMM_indices["var3"]


In [ ]:
def convert_time_to_ns(ds):
    # Original datetime64 coordinate
    datetime = ds["time"]

    # Compute nanoseconds since first timestep
    t0 = datetime.values[0]
    time_ns = (datetime.values - t0).astype("timedelta64[ns]").astype("timedelta64[ns]")

    new_datetime = datetime.expand_dims('batch').assign_coords(
        datetime=("time", datetime.values),   # secondary coordinate
        time=("time", time_ns)                # replace primary coordinate
    )

    # Assign new coordinates
    ds = ds.expand_dims('batch').assign_coords(
        datetime=new_datetime,
        time=("time", time_ns)
    )

    return ds

In [ ]:
start_date = '1992-08-14T00:00:00.000000000'
end_date = '1992-11-12T00:00:00.000000000'
yyyymm_strings = pd.date_range(
    pd.to_datetime(start_date).to_period("M").to_timestamp(),
    pd.to_datetime(end_date).to_period("M").to_timestamp(),
    freq="MS"
).strftime("%Y%m")


def pressure_level_preprocess(ds: xr.Dataset) -> xr.Dataset:
    # Subset to a specific region and keep only one variable
    subset_data = ds.sel(
        level=pressure_levels,
        time=ds.time.where(ds['time'].dt.hour.isin([0, 6, 12, 18]), drop=True)
    ).assign_coords({'level': pressure_levels.astype(np.int32)})

    target_grid = xr.Dataset(
        {
            "lat": (["lat"], np.arange(-90, 91, 1.0)),
            "lon": (["lon"], np.arange(0, 360, 1.0)),
        }
    )
    regridder = xe.Regridder(subset_data, target_grid, "bilinear", reuse_weights=False) 

    return regridder(subset_data) # type: ignore

pressure_level_base = "/gdex/data/d633000/e5.oper.an.pl"

pressure_levels = xr.DataArray(
    data = [50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000],
    dims=['level'],
    coords={'level': [50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000]}
)

pressure_level_variables = {
    "geopotential": "z",
    # "temperature": "t",
    # "u_component_of_wind": "u",
    # "v_component_of_wind": "v",
    # "specific_humidity": "q",
    # "vertical_velocity": "w",
}

pressure_level_variables_old_names = {
    "geopotential": "Z",
    "temperature": "T",
    "u_component_of_wind": "U",
    "v_component_of_wind": "V",
    "specific_humidity": "Q",
    "vertical_velocity": "W",
}

for variable in pressure_level_variables.keys():
    files_list = []

    logger.info(f"  {variable}")
    for ym in yyyymm_strings:
        pattern = f"{pressure_level_base}/{ym}/e5.oper.an.pl.*_{pressure_level_variables[variable]}.*.nc"
        files_list.extend(glob.glob(pattern))

    if not files_list:
        logger.info(f"No files found for variable {variable} in month {ym}")
    else:
        logger.info(f"    Loading files...")
        pressure_level_data = xr.open_mfdataset(sorted(files_list)[:3], preprocess=pressure_level_preprocess).load()


In [ ]:
# yyyymm_strings
pressure_level_data.coords['time']
# sorted(files_list)

# xr.open_dataset("/gdex/data/d633000/e5.oper.an.pl/199208/e5.oper.an.pl.128_130_t.ll025sc.1992080100_1992080123.nc")

In [ ]:
target_duration

In [ ]:
# plt.scatter(
#     RMM1.isel(time=slice(3371-100, 3371+100)),
#     RMM2.isel(time=slice(3371-100, 3371+100)),
# )
# plt.xlim(-4, 4)
# plt.ylim(-4, 4)
# plt.gca().set_aspect('equal')

# Configure plot
[fig, ax] = plt.subplots(figsize=(9,9))
plt.rcParams["axes.edgecolor"] = "black"
plt.rcParams["axes.linewidth"] = 3
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
plt.xlabel("RMM1")
plt.ylabel("RMM2")
ax.set_facecolor("white")

# Plot index points
colormap = sns.color_palette("viridis", as_cmap=True)
start_time = '2009-10-01T00:00:00.000000000'
end_time = '2010-04-30T00:00:00.000000000'
start_index = list(amplitude.time.values).index(amplitude.sel(time=start_time).time)
end_index = list(amplitude.time.values).index(amplitude.sel(time=end_time).time)
ax.plot(RMM1[start_index], RMM2[start_index], color="black", marker=".", ls="-", ms=30)
for i in range(start_index + 1, end_index + 1):
    ax.plot(
        RMM1[i],
        RMM2[i],
        color=colormap((i - start_index) / (end_index - start_index)),
        marker="o",
        ms=10,
    )

# Add phase regions overlay
circle1 = plt.Circle((0, 0), 1.0, color="k", fill=False, lw=3, zorder=10)
ax.hlines(y=0, xmin=-4, xmax=-1, color="k", lw=3, ls="-")
ax.hlines(y=0, xmin=1, xmax=4, color="k", lw=3, ls="-")
ax.vlines(x=0, ymin=-4, ymax=-1, color="k", lw=3, ls="-")
ax.vlines(x=0, ymin=1, ymax=4, color="k", lw=3, ls="-")
ax.plot([np.sqrt(2) / 2, 4], [np.sqrt(2) / 2, 4], color="k", lw=3, ls="-")


x_vals = {
    1:-1,
    2:-1,
    3:1,
    4:1,
    5:1,
    6:1,
    7:-1,
    8:-1
}

y_val1 = {
    1:0,
    2:-4,
    3:-4,
    4:0,
    5:0,
    6:np.linspace(0,4,len(RMM1)),
    7:np.linspace(0,4,len(RMM1)),
    8:0
}

y_val2 = {
    1:-1*np.linspace(0, 4, len(RMM1)),
    2:-1*np.linspace(0, 4, len(RMM1)),
    3:-1*np.linspace(0, 4, len(RMM1)),
    4:-1*np.linspace(0, 4, len(RMM1)),
    5:1*np.linspace(0, 4, len(RMM1)),
    6:4,
    7:4,
    8:1*np.linspace(0, 4, len(RMM1))
}

# Fill one of the phases in with blue
# val=2
# ax.fill_between(
#     x_vals[val]*np.linspace(0, 4, len(RMM1)), 
#     y_val1[val], 
#     y_val2[val]
# )

# x = np.linspace(-1,1,100)
# ax.fill_between(x, -np.sqrt(1-x**2), np.sqrt(1-x**2), color='white')

# Add lines to differentiate the phases
ax.plot([np.sqrt(2) / 2, 4], [-np.sqrt(2) / 2, -4], color="k", lw=3, ls="-")
ax.plot([-4, -np.sqrt(2) / 2], [4, np.sqrt(2) / 2], color="k", lw=3, ls="-")
ax.plot([-4, -np.sqrt(2) / 2], [-4, -np.sqrt(2) / 2], color="k", lw=3, ls="-")
ax.add_patch(circle1)

# Add phase labels
ax.text(-3.5, -0.26, f'Phase 1',  horizontalalignment='center',
     verticalalignment='center')
ax.text(-0.51, -3.75, f'Phase 2', horizontalalignment='center',
     verticalalignment='center')
ax.text(0.5, -3.75, f'Phase 3',   horizontalalignment='center',
     verticalalignment='center')
ax.text(3.5, -0.26, f'Phase 4',   horizontalalignment='center',
     verticalalignment='center')
ax.text(3.5, 0.25, f'Phase 5',    horizontalalignment='center',
     verticalalignment='center')
ax.text(0.5, 3.75, f'Phase 6',    horizontalalignment='center',
     verticalalignment='center')
ax.text(-0.51, 3.75, f'Phase 7',  horizontalalignment='center',
     verticalalignment='center')
ax.text(-3.5, 0.25, f'Phase 8',   horizontalalignment='center',
     verticalalignment='center')

# Add MJO-location labels
# ax.text(0, 3.5, f'Western Pacific',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

# ax.text(-3.3, 0, f'Western Hemisphere \n & Africa',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

# ax.text(3.3, 0, f'Maritime Continent',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

# ax.text(0, -3.5, f'Indian Ocean',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

ax.set_aspect("equal")
plt.tight_layout()

plt.show()

# Process ERA5 data

In [ ]:
# # start_time = "2021-05-01"
# # end_time = "2022-12-31"

# start_time = "1974-00-01"
# end_time = "1979-12-31"

# variable = 'zonal_wind'

# variable_name = {
#     'zonal_wind':'u_component_of_wind',
#     'meridional_wind':'v_component_of_wind'
#     }

# from datetime import datetime
# from dateutil.relativedelta import relativedelta

# def iter_year_months(start, end):
#     """
#     Yield (year, month) pairs for all months between two arbitrary dates.
#     Accepts datetime objects or ISO-format strings.
#     """

#     # Parse strings if needed
#     if isinstance(start, str):
#         start = datetime.fromisoformat(start)
#     if isinstance(end, str):
#         end = datetime.fromisoformat(end)

#     # Normalize to first-of-month
#     cursor = start.replace(day=1)
#     end_month = end.replace(day=1)

#     while cursor <= end_month:
#         yield cursor.year, cursor.month
#         cursor += relativedelta(months=1)

# import cdsapi

# c = cdsapi.Client()

# for year, month in iter_year_months(start_time, end_time):
#     print(year, month)
#     output_file = f"/glade/derecho/scratch/sressel/ECMWF/ERA5/daily_data/daily_25_degree_{variable}_{year}_{month}_raw.nc"
#     c.retrieve(
#         "reanalysis-era5-pressure-levels",
#         {
#             "product_type": "reanalysis",
#             "variable": [
#                 variable_name[variable],
#                 # "v_component_of_wind",
#             ],
#             "pressure_level": [
#                 "100", "125", "150", "175", "200", "225", "250", "300", "350", "400",
#                 "450", "500", "550", "600", "650", "700", "750", "775", "800", "825",
#                 "850", "875", "900", "925", "950", "975", "1000",
#             ],
#             "year": year,
#             "month": month,
#             "day": [
#                 "01","02","03","04","05","06","07","08","09","10",
#                 "11","12","13","14","15","16","17","18","19","20",
#                 "21","22","23","24","25","26","27","28","29","30","31",
#             ],
#             "time": ["00:00"],   # daily at 00 UTC
#             "area": [30, -180, -30, 180],  # North, West, South, East
#             "grid": [2.5, 2.5],  # 2.5° x 2.5°
#             "format": "netcdf",
#         },
#         output_file,
#     )

# # data_processed = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/ECMWF/ERA5/daily_data/daily_25_degree_{variable}_{start_time.year}_{end_time.year}_raw.nc").rename(
# #     {
# #         'longitude': 'lon',
# #         'latitude': 'lat',
# #         'valid_time': 'time',
# #         'pressure_level': 'plev',
# #     }
# # ).transpose("time", "plev", "lat", "lon").isel(plev=slice(None, None, -1)).sortby('plev').sortby('lat').coord_funcs.lon_to_360('lon')
# # data_processed.to_netcdf("/glade/u/home/sressel/spencer-scratch/ECMWF/ERA5/daily_data/daily_25_degree_{variable}_{start_time.year}_{end_time.year}.nc")


# Load Graphcast sample data

In [ ]:
# climatological_mean = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/climatological_mean_1990_2019.nc")
era5_data = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/2020-05_2021-02/era5_data.nc")
era5_data = era5_data.assign_coords(time=era5_data.datetime.isel(batch=0, drop=True)).drop_vars(["datetime", "number", "expver"])
# column_water_vapor = (1/9.8)*era5_data['specific_humidity'].integrate('level')

In [ ]:
event_start_date = np.datetime64("2020-12-28T00")
event_max_date = np.datetime64("2021-01-08T00")
event_end_date = np.datetime64("2021-02-14T00")

In [ ]:
(
    column_water_vapor.sel(time=slice(event_start_date, event_end_date)).sel(lat=slice(-20,20), lon=slice(60, 180))
     - column_water_vapor.sel(time=slice(event_start_date, event_end_date)).sel(lat=slice(-20,20), lon=slice(60, 180)).mean(dim=['lat', 'lon'])
).mean(dim='time').isel(batch=0, drop=True).plot.contourf(levels=21)

In [ ]:
three_day_accumulated_rainfall = (4*1000*era5_data['total_precipitation_6hr']).rolling(time=12, center=False).sum()/12
# from scipy.ndimage import gaussian_filter
smoothed_data = xr.zeros_like(three_day_accumulated_rainfall)
smoothed_data[:] = gaussian_filter(three_day_accumulated_rainfall, sigma=5, axes=[2, 3])
smoothed_data = smoothed_data.sel(lat=slice(-30, 30), drop=True)

In [ ]:
LPT_mask = smoothed_data > 12
LPT_masked = smoothed_data.where(LPT_mask)
lat2d, lon2d = xr.broadcast(LPT_masked.lat, LPT_masked.lon)

weights = np.cos(np.deg2rad(LPT_masked.lat))
weights2d, _ = xr.broadcast(weights, LPT_masked.lon)

centroid_lat_geo = xr.zeros_like(smoothed_data.isel(batch=0, lon=0, lat=0, drop=True))
centroid_lon_geo = xr.zeros_like(smoothed_data.isel(batch=0, lon=0, lat=0, drop=True))

from scipy.ndimage import label
min_area_km2 = 300000
R = 6371.0  # km
dlat = np.deg2rad(abs(float(LPT_masked.lat[1] - LPT_masked.lat[0])))
dlon = np.deg2rad(abs(float(LPT_masked.lon[1] - LPT_masked.lon[0])))
cell_area = (R**2) * dlat * dlon * weights2d  # varies with latitude

# --- tropical box: 15S-15N, 40E-160W ---
lat_min, lat_max = -15, 15
lon_min_360, lon_max_360 = 40, 200  # 40E to 160W, in 0-360 convention

# convert lon2d to 0-360 for the check regardless of the array's native convention
lon2d_360 = lon2d % 360

big_mask = xr.zeros_like(LPT_mask.isel(batch=0, drop=True))

for t in range(len(LPT_mask.time)):
    labels, nfeatures = label(LPT_mask.isel(time=t, batch=0).values)
    labels_da = xr.DataArray(
        labels,
        coords=LPT_masked.isel(time=t, batch=0).coords,
        dims=LPT_masked.isel(time=t, batch=0).dims,
    )

    region_ids = np.arange(1, nfeatures + 1)
    keep_ids = []

    for rid in region_ids:
        region_mask = labels_da == rid

        area = cell_area.values[labels == rid].sum()
        if area < min_area_km2:
            continue

        # geometric centroid of this region, area-weighted
        w_region = weights2d.where(region_mask)
        clat = float((lat2d * w_region).sum() / w_region.sum())
        clon360 = float((lon2d_360 * w_region).sum() / w_region.sum())

        in_lat_band = lat_min <= clat <= lat_max
        in_lon_band = lon_min_360 <= clon360 <= lon_max_360

        if in_lat_band and in_lon_band:
            keep_ids.append(rid)

    print(f"Kept {len(keep_ids)} of {nfeatures} regions")
    big_mask[t] = labels_da.isin(keep_ids)

    w_geo = weights2d.where(big_mask[t])
    centroid_lat_geo[t] = (lat2d * w_geo).sum() / w_geo.sum()
    centroid_lon_geo[t] = (lon2d * w_geo).sum() / w_geo.sum()

In [ ]:
masked_data = smoothed_data.where(big_mask)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import copy

def animate_precip_with_mask(
    precip_data,      # e.g. smoothed_data, dims (time, lat, lon) [+ batch]
    mask_data,         # e.g. big_mask, same spatial dims, boolean/0-1
    batch=0,
    vmin=10, vmax=1000,
    cmap="viridis",
    interval=200,       # ms between frames
    save_path=None,     # e.g. "animation.mp4" or "animation.gif",
    fade_frames=5
):
    # squeeze out batch dim if present
    if "batch" in precip_data.dims:
        precip_data = precip_data.isel(batch=batch)
    if "batch" in mask_data.dims:
        mask_data = mask_data.isel(batch=batch)

    ntime = precip_data.sizes["time"]
    lon = precip_data.lon.values
    lat = precip_data.lat.values
    mask_lon = mask_data.lon.values
    mask_lat = mask_data.lat.values

    times = pd.to_datetime(precip_data.time.values)

    # copy the colormap so set_under doesn't mutate the global registered cmap
    cmap_obj = copy.copy(plt.get_cmap(cmap))
    cmap_obj.set_under("white")

    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax, clip=False)

    fig, ax = plt.subplots(figsize=(10, 6))

    # initial frame
    im = ax.pcolormesh(
        lon, lat, precip_data.isel(time=0).values,
        cmap=cmap_obj, norm=norm, shading="auto"
    )
    cbar = fig.colorbar(im, ax=ax, label="Precipitation")

     # list of [contour_artist, current_alpha] pairs, oldest first
    contour_history = []

    def add_contour(t):
        cs = ax.contour(
            mask_lon, mask_lat, mask_data.isel(time=t).values.astype(float),
            levels=[0.5], colors="black", linewidths=1.5
        )
        cs.set_alpha(1.0)
        contour_history.append([cs, 1.0])

    def fade_and_prune():
        step = 1.0 / fade_frames
        still_alive = []
        for cs, alpha in contour_history:
            new_alpha = alpha - step
            if new_alpha <= 0:
                cs.remove()
            else:
                cs.set_alpha(new_alpha)
                still_alive.append([cs, new_alpha])
        contour_history[:] = still_alive

    def format_title(t):
        return times[t].strftime("%H UTC %d-%m-%Y")

    add_contour(0)
    title = ax.set_title(format_title(0))

    def update(t):
        im.set_array(precip_data.isel(time=t).values.ravel())
        if t > 0:
            fade_and_prune()
            add_contour(t)
        title.set_text(format_title(t))
        return im,

    anim = animation.FuncAnimation(
        fig, update, frames=ntime, interval=interval, blit=False
    )

    if save_path:
        anim.save(save_path, writer="pillow" if save_path.endswith(".gif") else "ffmpeg")

    plt.close(fig)
    return anim

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import copy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def animate_precip_with_mask(
    precip_data,
    mask_data,
    batch=0,
    vmin=10, vmax=100,
    cmap="viridis",
    interval=200,
    save_path=None,
    track_cmap="plasma",
):
    if "batch" in precip_data.dims:
        precip_data = precip_data.isel(batch=batch)
    if "batch" in mask_data.dims:
        mask_data = mask_data.isel(batch=batch)

    ntime = precip_data.sizes["time"]

    precip_lon = precip_data.lon.values
    precip_lat = precip_data.lat.values
    mask_lon = mask_data.lon.values
    mask_lat = mask_data.lat.values

    times = pd.to_datetime(precip_data.time.values)

    cmap_obj = copy.copy(plt.get_cmap(cmap))
    cmap_obj.set_under("white")
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax, clip=False)

    track_cmap_obj = plt.get_cmap(track_cmap)
    track_norm = mcolors.Normalize(vmin=0, vmax=max(ntime - 1, 1))

    # --- set up cartopy axes ---
    proj = ccrs.PlateCarree(central_longitude=-180)
    data_crs = ccrs.PlateCarree()
    fig, ax = plt.subplots(
        figsize=(10, 6), subplot_kw={"projection": proj}
    )

    # land underneath the data, drawn at low zorder
    # ax.add_feature(cfeature.LAND, facecolor="lightgray", zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=3)
    # ax.add_feature(cfeature.BORDERS, linewidth=0.4, linestyle=":", zorder=3)

    # set extent to match your data domain (with a small pad)
    ax.set_extent(
        [50, 210, precip_lat.min(), precip_lat.max()],
        crs=data_crs,
    )

    im = ax.pcolormesh(
        precip_lon, precip_lat, precip_data.isel(time=0).values,
        cmap=cmap_obj, norm=norm, shading="auto",
        transform=data_crs, zorder=1,
    )
    cbar = fig.colorbar(im, ax=ax, label="Precipitation", extend="min")

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5, zorder=4)
    gl.top_labels = False
    gl.right_labels = False

    contour_history = []

    def add_contour(t):
        color = track_cmap_obj(track_norm(t))
        cs = ax.contour(
            mask_lon, mask_lat, mask_data.isel(time=t).values.astype(float),
            levels=[0.5], colors=[color], linewidths=1.5,
            transform=data_crs, zorder=2,
        )
        contour_history.append(cs)

    def format_title(t):
        return times[t].strftime("%H UTC %d-%m-%Y")

    add_contour(0)
    title = ax.set_title(format_title(0))

    def update(t):
        im.set_array(precip_data.isel(time=t).values.ravel())
        if t > 0:
            add_contour(t)
        title.set_text(format_title(t))
        return im,

    anim = animation.FuncAnimation(
        fig, update, frames=ntime, interval=interval, blit=False
    )

    if save_path:
        anim.save(save_path, writer="pillow" if save_path.endswith(".gif") else "ffmpeg")

    plt.close(fig)
    return anim

In [ ]:
anim = animate_precip_with_mask(
    three_day_accumulated_rainfall.sel(lat=slice(-30, 30)).sel(time=slice(event_start_date-np.timedelta64(5, 'D'), event_end_date+np.timedelta64(5, 'D'))), 
    big_mask.sel(time=slice(event_start_date-np.timedelta64(5, 'D'), event_end_date+np.timedelta64(5, 'D'))), 
    batch=0
)

# in a Jupyter notebook, display inline:
from IPython.display import HTML
HTML(anim.to_jshtml())

In [ ]:
smoothed_data.sel(lat=slice(-15,15), lon=slice(40, 180), time=slice(event_start_date-np.timedelta64(10, 'D'), event_end_date+np.timedelta64(30, 'D'))).mean(dim='lat').plot()
centroid_lon_geo.sel(time=slice(event_start_date-np.timedelta64(10, 'D'), event_end_date+np.timedelta64(30, 'D'))).plot(y='time', color='k')

## Optimized initial condition

### Load optimized data

In [ ]:
import re
from pathlib import Path
from datetime import datetime

def find_era5_data(target_date: str, parent_dir: str, filename: str = "era5_data.nc") -> Path:
    """
    Locate the ERA5 data file whose containing folder (named YYYY-MM_YYYY-MM)
    spans the given target date.

    Parameters
    ----------
    target_date : str
        Date in 'YYYY-MM-DD' format (time component, if present, is ignored).
    parent_dir : str
        Directory containing the YYYY-MM_YYYY-MM subfolders.
    filename : str
        Name of the file to locate within the matching folder (default: 'era5_data.nc').

    Returns
    -------
    Path
        Full path to the matching data file.

    Raises
    ------
    ValueError
        If target_date can't be parsed, no folder matches, multiple folders
        match, or the expected file doesn't exist in the matched folder.
    """
    # Parse just the date portion (drop any 'T...' time component if present)
    date_str = target_date.split("T")[0]
    try:
        target = datetime.strptime(date_str, "%Y-%m-%d")
    except ValueError as e:
        raise ValueError(f"Invalid date '{target_date}': {e}")

    target_ym = target.year * 100 + target.month  # e.g. 202010

    pattern = re.compile(r"^(\d{4})-(\d{2})_(\d{4})-(\d{2})$")
    parent = Path(parent_dir)

    matches = []
    for entry in parent.iterdir():
        if not entry.is_dir():
            continue
        m = pattern.match(entry.name)
        if not m:
            continue

        start_ym = int(m.group(1)) * 100 + int(m.group(2))
        end_ym = int(m.group(3)) * 100 + int(m.group(4))

        if start_ym <= target_ym <= end_ym:
            matches.append(entry)

    if not matches:
        raise ValueError(f"No folder in {parent_dir} spans {date_str}")
    if len(matches) > 1:
        raise ValueError(
            f"Multiple folders span {date_str}: {[str(m) for m in matches]}. "
            "Resolve the overlap before proceeding."
        )

    data_file = matches[0] / filename
    if not data_file.is_file():
        raise ValueError(f"Matched folder {matches[0]}, but {data_file} does not exist")

    return data_file

In [ ]:
def format_graphcast_output(ds):
    return ds.assign_coords({'time':ds.datetime.isel(batch=0, drop=True)}).isel(batch=0, drop=True).drop_vars(["number", "expver", "datetime"])

In [ ]:
initial_condition_date = "2021-01-01T00"
optimization_timesteps = 40
run_notes = "vpm_test_2"

optimized_initial_condition = xr.open_dataset(f"/glade/u/home/sressel/spencer-scratch/graphcast_input_data/optimized_initial_conditions/{initial_condition_date}/optimized_initial_condition_{optimization_timesteps}_{run_notes}.nc")
optimized_initial_condition = format_graphcast_output(optimized_initial_condition)
optimized_initial_condition['Column Water Vapor'] = (100/9.8)*optimized_initial_condition['specific_humidity'].integrate('level')

era5_initial_condition_filepath = find_era5_data(initial_condition_date, config.INPUT_DATA_DIRECTORY)
era5_initial_condition = xr.open_dataset(f"{era5_initial_condition_filepath}")
era5_initial_condition = format_graphcast_output(era5_initial_condition)
era5_initial_condition['Column Water Vapor'] = (100/9.8)*era5_initial_condition['specific_humidity'].integrate('level')

In [ ]:
time_to_plot = initial_condition_date
level_to_plot = 850
variable_to_plot = 'Column Water Vapor'

# [fig, ax] = plt.subplots(3, 1, figsize=(12,16))

fig, ax = plt.subplots(nrows=3,ncols=1,
                        subplot_kw={'projection': ccrs.PlateCarree()},
                        figsize=(11,8.5))

i=0
c_opt = ax[0].contourf(
    optimized_initial_condition[variable_to_plot].lon,
    optimized_initial_condition[variable_to_plot].lat,
    optimized_initial_condition[variable_to_plot].coord_funcs.sel(level=level_to_plot, time=time_to_plot),
    levels = 21
)
fig.colorbar(c_opt, ax=ax[0])
ax[0].add_feature(cfeature.COASTLINE, linewidth=0.5)

c_init = ax[1].contourf(
    era5_initial_condition.lon,
    era5_initial_condition.lat,
    era5_initial_condition[variable_to_plot].coord_funcs.sel(level=level_to_plot, time=time_to_plot),
    levels = c_opt.levels
)
fig.colorbar(c_init, ax=ax[1])
ax[1].add_feature(cfeature.COASTLINE, linewidth=0.5)

c_diff = ax[2].contourf(
    era5_initial_condition.lon,
    era5_initial_condition.lat,
    (
        optimized_initial_condition[variable_to_plot].coord_funcs.sel(level=level_to_plot, time=time_to_plot)
        - era5_initial_condition[variable_to_plot].coord_funcs.sel(level=level_to_plot, time=time_to_plot)
    ),
    # levels = np.linspace(-0.01, 0.01, 21),
    levels=21,
    cmap = 'coolwarm'
)
fig.colorbar(c_diff, ax=ax[2])
ax[2].add_feature(cfeature.COASTLINE, linewidth=0.5)

# for axis in ax:
#     rect = patches.Rectangle(
#         (70, -10),        # lower-left corner
#         30, 20,   # rectangle size
#         linewidth=1.5,
#         edgecolor='red',
#         facecolor='none',   # or a color like 'lightgray'
#         alpha=0.5
#     )

#     axis.add_patch(rect)

# for axis in ax:
#     axis.set_xlim(60, 110)
#     axis.set_ylim(-15, 15)

plt.show()

In [ ]:
def recalculate_timestamps(data, init_date):
    init_time = data.where(data.datetime == np.datetime64(init_date), drop=True).time
    new_times = data.time.values - init_time.values
    retimed_data = data.assign_coords({'time':new_times})
    return retimed_data

# def calculate_velocity_potential_indices(data, init_date):

"""
    Project GraphCast-style forecast/target data onto pre-computed velocity
    potential mode (VPM) EOF structures to compute VPM1 and VPM2 indices,
    following the methodology of Ventrice et al. (2013),
    https://doi.org/10.1175/MWR-D-12-00327.1.

    The input data is processed into standardized anomalies (meridional
    average, climatology removal, annual cycle removal, 120-day running-mean
    removal, and normalization by variable-specific standard deviations) for
    three fields — 200-hPa velocity potential (VP200), 200-hPa zonal wind
    (U200), and 850-hPa zonal wind (U850) — then projected onto pre-computed
    EOF patterns via a dot product to yield the leading two principal
    components (PCs), which are rescaled into VPM1 and VPM2.

    All climatology, annual-cycle, standard-deviation, and EOF reference
    fields are pre-computed from 6-hourly (resampled to daily), 1-degree ERA5 reanalysis data
    spanning 1990-2019, and loaded from fixed paths on GLADE.

    Parameters
    ----------
    data : xarray.Dataset
        GraphCast-style dataset containing at least the following variables:
        - 'u_component_of_wind', with a 'level' coordinate including 200
        and 850 hPa
        - 'v_component_of_wind', with a 'level' coordinate including 200 hPa
        (used internally by `calculate_velocity_potential` to compute
        VP200)
        Must also have 'time' and 'lat' coordinates spanning at least
        15°S-15°N and enough time history (120+ days) prior to the period
        of interest for the running-mean removal step to be valid.

    Returns
    -------
    VPM1 : xarray.DataArray
        Leading VPM principal component, standardized by its own time std.
    VPM2 : xarray.DataArray
        Second VPM principal component, standardized by [mode 1's] time std.
        (NOTE: see Warnings below — this may not be intentional.)

    Notes
    -----
    Reference data loaded from GLADE (fixed paths, 1990-2019 ERA5 basis):
    - Climatological mean: climatological_mean_1990_2019.nc
    - Annual cycle regression coefficients: annual_cycle_regression_coefficients_1990_2019.nc
    - Standard deviations: standard_deviations_1990_2019.nc
    - EOF patterns: EOFs_for_projection_1990_2019.nc
    """

# Load in climatological mean data
climatological_mean_data = xr.open_dataset(
    "/glade/u/home/sressel/spencer-scratch/graphcast_input_data/climatological_mean_1990_2019.nc"
)

# Load in regression coefficients to reconstruct the annual cycle
from auxiliary_functions.annual_cycle_functions import reconstruct_annual_cycle
annual_cycle_regression_coefficients = xr.open_dataset(
    "/glade/u/home/sressel/spencer-scratch/graphcast_input_data/annual_cycle_regression_coefficients_1990_2019.nc"
)

# Load in zonal-mean standard deviations from EOF computation to stanrdardize variables
standard_deviations = xr.open_dataset(
    "/glade/u/home/sressel/spencer-scratch/graphcast_input_data/standard_deviations_1990_2019.nc"
)

# Load in the pre-computed EOFs that the data will be projected onto
EOFs = xr.open_dataset(
    "/glade/u/home/sressel/spencer-scratch/graphcast_input_data/EOFs_for_projection_1990_2019.nc"
)

# Concatenate the EOFs together along the 'lon' dimension
EOFs_concatenated = xr.concat(
    [EOFs['VP200'], EOFs['U200'], EOFs['U850']],
    dim='lon'
)

init_date = initial_condition_date

# Load in lookback data
lookback_data_filepath = find_era5_data(init_date, "/glade/u/home/sressel/spencer-scratch/graphcast_input_data")
lookback_data = xr.open_dataset(lookback_data_filepath)
retimed_lookback_data = recalculate_timestamps(lookback_data, init_date).sel(time=slice(None, np.timedelta64(0)))

data = test_data

# Append the graphcast predictions to the lookback data
data_appended = xr.concat(
    [retimed_lookback_data, data],
    dim='time'
)

decoded_data_time = np.datetime64(init_date)  + data_appended["time"].values.astype("timedelta64[ns]")
data_with_date = data_appended.assign_coords(time=decoded_data_time).resample({'time':'1D'}).mean(dim='time')

# Extract relevant levels from input data
U200 = data_with_date.sel(level=200)
U850 = data_with_date.sel(level=850)

# Calculate velocity potential for the predictions and targets
VP200 = calculate_velocity_potential_era5(data)

# Loop over each of the three variables to process them
VPM_variables = {'VP200':VP200, 'U200':U200, 'U850':U850}
data_standardized = {}
for variable in ['VP200', 'U200', 'U850']:

    # Meridionally averaged over 15°S-15°N
    data_meridionally_averaged = VPM_variables[variable].sel(lat=slice(-15,15)).mean(dim='lat')

    # Remove the 1990-2019 time mean climatology
    data_climatological_anomaly = data_meridionally_averaged - climatological_mean_data[variable]

    # Reconstruct the 1990-2019 annual cycle & remove it from the data
    data_annual_cycle = reconstruct_annual_cycle(
        data_climatological_anomaly.time, 
        annual_cycle_regression_coefficients[variable],
        np.datetime64(annual_cycle_regression_coefficients.attrs['t0'])
    )
    data_deannualized = data_climatological_anomaly - data_annual_cycle

    # Remove the 120-day rolling mean
    data_anomalies = (
        data_deannualized
        - data_deannualized.rolling(time=120, center=False).mean()
    )
    data_standardized[variable] = data_anomalies / standard_deviations[variable]

# Concatenate the variables along the 'lon' axis 
data_concatenated = xr.concat(
    [data_standardized[v] for v in ['VP200', 'U200', 'U850']],
    dim='lon'
)

# Project the processed variables onto the pre-computed EOF patterns to calculate PCs
projected_PCs = xr.dot(
    EOFs_concatenated,
    data_concatenated,
    dim='lon'
)

# Rescale the PCs by the first mode std. to get VPM1 and VPM2
VPM1 = projected_PCs.sel(mode=1) / projected_PCs.std(dim='time').sel(mode=1)
VPM2 = projected_PCs.sel(mode=2) / projected_PCs.std(dim='time').sel(mode=1)


def calculate_velocity_potential(data):

    from windspharm.xarray import VectorWind

    U200 = data['u_component_of_wind'].sel(level=200)
    V200 = data['v_component_of_wind'].sel(level=200)

    # U200_batch = U200.isel(batch=0, drop=True)
    # V200_batch = V200.isel(batch=0, drop=True)

    _, VP200_vals = VectorWind(U200, V200).sfvp()

    VP200 = xr.zeros_like(U200)
    VP200[0] = VP200_vals.isel(lat=slice(None, None, -1)).values
    VP200.name = 'velocity_potential'

    return VP200


In [ ]:
time_to_plot = initial_condition_date
level_to_plot = 850
variable_to_plot = 'specific_humidity'

SAVE_FIG = True
PLOT_MODE = "slides"
FIG_LAYOUT = 'full'
plt.style.use('bmh')
set_plot_mode(PLOT_MODE)
plt.rcParams['mathtext.fontset'] = 'dejavusans'
plt.rcParams['font.size'] = '12'
plt.rcParams['axes.titlesize'] = '12'
plt.rcParams['axes.labelsize'] = '12'
plt.rcParams['xtick.labelsize'] = '10'
plt.rcParams['ytick.labelsize'] = '10'
plt.rcParams['lines.linewidth'] = '1'

# fig = plt.figure(figsize=get_figsize(PLOT_MODE, FIG_LAYOUT))
fig = plt.figure(figsize=(4,7))
# 3 rows x 2 cols: left column for maps, right (narrow) column for colorbars.
# width_ratios makes the colorbar column slim relative to the map column.
gs = GridSpec(
    nrows=3, ncols=2,
    width_ratios=[1, 0.03],
    height_ratios=[1, 1, 1],
    wspace=0.05, hspace=0.35
)

axes = []
axes.append(fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree()))
axes.append(fig.add_subplot(gs[1, 0], projection=ccrs.PlateCarree()))
axes.append(fig.add_subplot(gs[2, 0], projection=ccrs.PlateCarree()))

cbar_axes = []
cbar_axes.append(fig.add_subplot(gs[0:2, 1]))
cbar_axes.append(fig.add_subplot(gs[2, 1]))

# --- Row 1: ERA5 IC (same levels/colorbar as row 0) ---
axes[0].set_title("a) ERA5 Initial Condition", loc='left')
c_init = axes[0].contourf(
    era5_initial_condition.lon,
    era5_initial_condition.lat,
    era5_initial_condition[variable_to_plot].sel(level=level_to_plot, time=time_to_plot),
    levels=c_opt.levels
)
axes[0].add_feature(cfeature.COASTLINE, linewidth=0.5)

# --- Row 0: optimized IC ---
axes[1].set_title("b) Graphcast Optimized\n Initial Condition", loc='left')
c_opt = axes[1].contourf(
    optimized_initial_condition[variable_to_plot].lon,
    optimized_initial_condition[variable_to_plot].lat,
    optimized_initial_condition[variable_to_plot].sel(level=level_to_plot, time=time_to_plot),
    levels=21
)
axes[1].add_feature(cfeature.COASTLINE, linewidth=0.5)

# --- Joint colorbar spanning rows 0 and 1, on the right ---
fig.colorbar(c_opt, cax=cbar_axes[0])

# --- Row 2: difference plot, own colorbar ---
axes[2].set_title("c) Difference", loc='left')
c_diff = axes[2].contourf(
    era5_initial_condition.lon,
    era5_initial_condition.lat,
    (
        optimized_initial_condition[variable_to_plot].sel(level=level_to_plot, time=time_to_plot)
        - era5_initial_condition[variable_to_plot].sel(level=level_to_plot, time=time_to_plot)
    ),
    levels=21,
    cmap='coolwarm'
)
axes[2].add_feature(cfeature.COASTLINE, linewidth=0.5)

fig.colorbar(c_diff, cax=cbar_axes[1])

for i, axis in enumerate(axes):
    axis.set_xlim(50, 70)
    axis.set_ylim(-20, 0)

    gl = axis.gridlines(
        draw_labels=True,
        linewidth=0.5,
        color='gray',
        alpha=0.5,
        linestyle='--'
    )
    gl.top_labels = False
    gl.right_labels = False
    # Only show y-axis (lat) labels on the leftmost column — all axes are
    # already in column 0, so this line is a no-op here, but keeps the
    # pattern ready if you ever add more columns.
    if i < len(ax) - 1:
        gl.bottom_labels = False  # avoid redundant lon labels between stacked rows

    axis.set_aspect('equal')

fig.suptitle(
    (
        f"{np.datetime64(time_to_plot).astype('M8[ms]').astype('O').strftime('%HZ %d-%^b-%Y')}"
        + f"\n{config.VARIABLE_SHORTNAMES[variable_to_plot]}" 
        + (f"{level_to_plot}" if 'level' in era5_initial_condition[variable_to_plot].coords else "")
    ),
    x=0.5, y=1.01, ha="center"
)


# Save figure
output_filename = (
    f"{np.datetime64(time_to_plot).astype('M8[ms]').astype('O').strftime('%Y-%m-%dT%H')}"
    + f"_{config.VARIABLE_SHORTNAMES[variable_to_plot]}"
    + (f"{level_to_plot}" if 'level' in era5_initial_condition[variable_to_plot].coords else "")
    + f"_{run_notes}"
)
logger.info(f"Output directory: {config.OUTPUT_DIRECTORY}/era5_control_perfect_comparisons/")
logger.info(f"Output filename: {output_filename}")
if SAVE_FIG:
    logger.info(f"Saving...")
    plt.savefig(
        f"{config.OUTPUT_DIRECTORY}/{output_filename}.svg",
        dpi=500,
        bbox_inches="tight",
    )
else:
    logger.info("Not Saving")

logger.info("Finished")
plt.show()

In [ ]:
# era5_data['specific_humidity'].sel(level=850, lat=slice(-30,30), lon=slice(60,180), datetime="2020-12-31T18").plot(levels=np.linspace(0, 0.015, 11), cmap='viridis', extend='max')
# plt.figure()
# optimized_initial_condition['specific_humidity'].sel(level=850, lat=slice(-30,30), lon=slice(60,180)).isel(time=0, batch=0).plot(levels=np.linspace(0, 0.015, 11), cmap='viridis', extend='max')
time_to_plot = "2021-01-01T00"
variable_to_plot = 'specific_humidity'
unit_multiplier = (1000)
# unit_multiplier = 1

SAVE_FIG = False
PLOT_MODE = "slides"
FIG_LAYOUT = 'full'
plt.style.use('bmh')
set_plot_mode(PLOT_MODE)
plt.rcParams['mathtext.fontset'] = 'dejavusans'
plt.rcParams['font.size'] = '14'
plt.rcParams['axes.labelsize'] = '14'
plt.rcParams['xtick.labelsize'] = '12'
plt.rcParams['ytick.labelsize'] = '12'
plt.rcParams['lines.linewidth'] = '2'

fig = plt.figure(figsize=get_figsize(PLOT_MODE, FIG_LAYOUT))

gs = GridSpec(2, 2, figure=fig, width_ratios=[30,1])
gs.update(right=0.6, hspace=0.3, wspace=0.05)

axes = [
    fig.add_subplot(gs[0,0], projection=ccrs.PlateCarree(central_longitude=-180)),
    fig.add_subplot(gs[1,0], projection=ccrs.PlateCarree(central_longitude=-180)),
]
cbar_axes = [
    fig.add_subplot(gs[:, -1]),
]
data_crs = ccrs.PlateCarree()

axes[0].set_title("a) ERA5 Data", loc='left')
im = axes[0].contourf(
    era5_initial_condition['lon'],
    era5_initial_condition['lat'],
    1000*era5_initial_condition['specific_humidity'].coord_funcs.sel(level=850, time=time_to_plot),
    transform=data_crs,
    levels=np.linspace(0, 15, 11),
    cmap='viridis',
    extend='max'
)
fig.colorbar(im, cax=cbar_axes[0])

axes[1].set_title("b) Optimized Data", loc='left')
axes[1].contourf(
    optimized_initial_condition['lon'],
    optimized_initial_condition['lat'],
    1000*optimized_initial_condition['specific_humidity'].coord_funcs.sel(level=850, time=time_to_plot),
    transform=data_crs,
    levels=np.linspace(0, 15, 11),
    cmap='viridis',
    extend='max'
)
for axis in axes:
    axis.add_feature(cfeature.COASTLINE, linewidth=0.5)
    axis.set_aspect('equal')
    gl = axis.gridlines(crs=data_crs, draw_labels=True,
                    linewidth=1.5, color='#bcbcbc', alpha=0.5, linestyle='-')
    gl.left_labels = False
    gl.top_labels = False # Hides x-labels on the bottom
    gl.xlocator = mticker.FixedLocator([60, 90, 120, 150, 180])
    gl.ylocator = mticker.FixedLocator([-30, -15, 0, 15, 30])
    gl.xformatter = LongitudeFormatter()
    gl.yformatter = LatitudeFormatter()
    axis.set_extent([60, 180, -30, 30])

fig.suptitle(
    f"{np.datetime64(time_to_plot).astype('M8[ms]').astype('O').strftime('%d-%^b-%Y')}",
    x=axes[0].get_position().x0 + 0.5 * axes[0].get_position().width,
    y=1.005,
    ha="center"
)

plt.show()

# Analyze predicted data

In [ ]:
logger.info("Load Graphcast Prediction Data")

# --- Run configuration ---
# prediction_initial_date = "2003-11-04T00"
# prediction_initial_date = "2020-09-24T00"
prediction_initial_date = "2020-12-13T00"
# prediction_initial_date = "2021-01-01T00"
prediction_timesteps = 60
# run_notes = "_zhao_2013"
# run_notes = "_vpm_test_3"
run_notes = "_ventrice_VPM_test"

logger.info(f"Initial Date: {prediction_initial_date}, {prediction_timesteps//config.TIMESTEPS_PER_DAY} day prediction, {run_notes[1:]}")

vars_to_drop = [
    "number",
    "expver",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "2m_temperature",
    "geopotential",
    "mean_sea_level_pressure",
    "specific_humidity",
    "temperature",
    "total_precipitation_6hr",
    "vertical_velocity",

]

predicted_data = {}
for data_source in ['control', 'perfect']:
# --- control model forecast --
    logger.info(f"Loading Graphcast '{data_source.title()}' Predictions...")
    prediction_filepath = (
        f"{config.GRAPHCAST_DATA_DIRECTORY}/{data_source}_model_forecasts/{prediction_initial_date}/{prediction_initial_date}_{prediction_timesteps}{run_notes}.zarr"
    )
    init_date = prediction_filepath.split("/")[-1].split("_")[0]

    predictions = xr.open_zarr(prediction_filepath, drop_variables=vars_to_drop).load().sel(batch=0, drop=True)
    datetimes = convert_ns_to_datetime(predictions.time, init_date)
    predicted_data[data_source.title()] = predictions.assign_coords({'time':datetimes.datetime}).drop_vars(["datetime"])

graphcast_control_predictions = predicted_data['Control']
graphcast_perfect_predictions = predicted_data['Perfect']

# --- Perfect model forecast ---
# logger.info("Loading Graphcast 'Perfect' Predictions...")
# perfect_prediction_filepath = f"{config.GRAPHCAST_DATA_DIRECTORY}/perfect_model_forecasts/{prediction_initial_date}/{prediction_initial_date}_{prediction_timesteps}{run_notes}.zarr"
# graphcast_perfect_predictions = xr.open_zarr(perfect_prediction_filepath).load().sel(batch=0, drop=True).drop_vars(['number', 'expver'])
# datetimes = convert_ns_to_datetime(graphcast_perfect_predictions.time, init_date)
# graphcast_perfect_predictions = graphcast_perfect_predictions.assign_coords({'time':datetimes.datetime}).drop_vars(['datetime'])

# --- Initial conditions ---
logger.info("Loading ERA5 data...")
era5_initial_conditions_filepath = str(graphcast_control_predictions['initial_conditions_path'].values)
era5_data = xr.open_dataset(era5_initial_conditions_filepath, drop_variables=vars_to_drop+["land_sea_mask", "geopotential_at_surface", "toa_incident_solar_radiation"])
era5_data = era5_data.assign_coords(time=era5_data.datetime.isel(batch=0, drop=True)).drop_vars(['datetime'])
era5_data = era5_data.sel(
    time=slice(
        np.datetime64(init_date)+np.timedelta64(6, "h"),
        np.datetime64(init_date)+np.timedelta64(6*prediction_timesteps, "h"),
    )
)

logger.info("Finished")

In [ ]:
ref_u = era5_data['u_component_of_wind'].sel(level=slice(700, 1000)).integrate('level')

import jax
jax.config.update("jax_enable_x64", True)
test_u = jax.scipy.integrate.trapezoid(
    y=era5_data['u_component_of_wind'].sel(level=slice(700, 1000)).values.astype(np.float64),
    x=era5_data['u_component_of_wind'].sel(level=slice(700, 1000)).level.values.astype(np.float64),
    axis=2,
)

In [ ]:
print(era5_data['u_component_of_wind'].dtype)        # is the source data itself float32?
print(era5_data['u_component_of_wind'].level.dtype)   # and the coordinate?

import jax.numpy as jnp
print(jnp.array([1.0]).dtype)

In [ ]:
plt.contourf(test_u[0][0])

In [ ]:
graphcast_control_predictions['Column Water Vapor'] = (100/9.8)*graphcast_control_predictions['specific_humidity'].integrate('level')
graphcast_perfect_predictions['Column Water Vapor'] = (100/9.8)*graphcast_perfect_predictions['specific_humidity'].integrate('level')
era5_data['Column Water Vapor'] = (100/9.8)*era5_data['specific_humidity'].integrate('level')

In [ ]:
import sys
sys.path.append("/glade/u/home/sressel/thesis-work/python/graphcast/gc-initial-condition-optimization/batch_modules")
from vpm_loss_utils import load_data_for_vpm 
VPM_datasets = load_data_for_vpm(init_date, prediction_timesteps)

In [ ]:
reference_VPM = xr.open_dataarray("/glade/u/home/sressel/spencer-scratch/ventrice_2013_VPM.nc")

from auxiliary_functions.calculate_velocity_potential import VelocityPotentialNumpy
def calculate_velocity_potential_indices_numpy(data, init_date, VPM_datasets):

    """
        Project GraphCast-style forecast/target data onto pre-computed velocity
        potential mode (VPM) EOF structures to compute VPM1 and VPM2 indices,
        following the methodology of Ventrice et al. (2013),
        https://doi.org/10.1175/MWR-D-12-00327.1.

        The input data is processed into standardized anomalies (meridional
        average, climatology removal, annual cycle removal, 120-day running-mean
        removal, and normalization by variable-specific standard deviations) for
        three fields — 200-hPa velocity potential (VP200), 200-hPa zonal wind
        (U200), and 850-hPa zonal wind (U850) — then projected onto pre-computed
        EOF patterns via a dot product to yield the leading two principal
        components (PCs), which are rescaled into VPM1 and VPM2.

        All climatology, annual-cycle, standard-deviation, and EOF reference
        fields are pre-computed from 6-hourly (resampled to daily), 1-degree ERA5 reanalysis data
        spanning 1990-2019, and loaded from fixed paths on GLADE.

        This is a plain numpy/xarray port of the JAX version -- same
        computation, but operates on ordinary xarray.DataArrays backed by
        numpy arrays instead of xarray_jax-wrapped jax arrays, so it can be
        used outside a jax pipeline (no jit/grad support, but no jax
        dependency either).

        Parameters
        ----------
        data : xarray.Dataset
            GraphCast-style dataset containing at least the following variables:
            - 'u_component_of_wind', with a 'level' coordinate including 200
            and 850 hPa
            - 'v_component_of_wind', with a 'level' coordinate including 200 hPa
            (used internally by `calculate_velocity_potential` to compute
            VP200)
            Must also have 'time' and 'lat' coordinates spanning at least
            15°S-15°N and enough time history (120+ days) prior to the period
            of interest for the running-mean removal step to be valid.

        Returns
        -------
        VPM1 : xarray.DataArray
            Leading VPM principal component, standardized by its own time std.
        VPM2 : xarray.DataArray
            Second VPM principal component, standardized by [mode 1's] time std.

        Notes
        -----
        Reference data loaded from GLADE (fixed paths, 1990-2019 ERA5 basis):
        - Climatological mean: climatological_mean_1990_2019.nc
        - Annual cycle regression coefficients: annual_cycle_regression_coefficients_1990_2019.nc
        - Standard deviations: standard_deviations_1990_2019.nc
        - EOF patterns: EOFs_for_projection_1990_2019.nc
        """

    # Loop over each of the three variables to process them
    VPM_variables = {'VP200': data['VP200'], 'U200': data['U200'], 'U850': data['U850']}
    data_standardized = {}

    for variable in ['VP200', 'U200', 'U850']:

        # Meridionally averaged over 15°S-15°N
        data_meridionally_averaged = VPM_variables[variable].sel(lat=slice(-15, 15)).mean(dim='lat')

        # Remove the 1990-2019 time mean climatology
        data_climatological_anomaly = data_meridionally_averaged - VPM_datasets['Climatological Mean'][variable]

        data_deannualized = data_climatological_anomaly - VPM_datasets['Annual Cycle'][variable]
        print(data_deannualized.shape)

        def trailing_moving_average(x, n=120):

            ret = np.cumsum(x, axis=0)

            ret[n:] = ret[n:] - ret[:-n]

            valid = ret[n - 1:] / n

            # Add n-1 NaN entries at the beginning
            padding_shape = list(x.shape)
            padding_shape[0] = n - 1

            padding = np.full(
                padding_shape,
                np.nan,
                dtype=x.dtype,
            )

            return np.concatenate([padding, valid], axis=0)

        rolling_mean = trailing_moving_average(np.asarray(data_deannualized.data))
        data_anomaly_values = data_deannualized.data - rolling_mean

        data_anomalies = xr.DataArray(
            data_anomaly_values,
            dims=data_deannualized.dims,
            coords=data_deannualized.coords,
            name=data_deannualized.name
        ).isel(time=slice(119, None))

        data_standardized[variable] = data_anomalies / VPM_datasets['EOF Standard Deviations'][variable]

    # Concatenate the variables along the 'lon' axis
    data_concatenated = xr.concat(
        [data_standardized[v] for v in ['VP200', 'U200', 'U850']],
        dim='lon'
    )

    # Project the processed variables onto the pre-computed EOF patterns to calculate PCs
    projected_PCs = xr.dot(
        VPM_datasets['EOFs'].assign_coords({'lon':np.arange(0, len(VPM_datasets['EOFs'].lon), 1)}),
        data_concatenated.assign_coords({'lon':np.arange(0, len(data_concatenated.lon), 1)}),
        dim='lon'
    )

    # Rescale the PCs by the first mode std. to get VPM1 and VPM2
    VPM = projected_PCs / VPM_datasets['PC Standard Deviation']

    # Only keep the time slices that are after the init_date (the lookback data is not part of the loss)
    return VPM.sel(time=slice(np.datetime64(init_date), None))

In [ ]:
concatenated_data = {}
VPM = {}

vp = VelocityPotentialNumpy(nlat=181, nlon=360, truncation=42)

lookback_data = VPM_datasets['Lookback Data']

for dataset, data in zip(
    ["Control", "Perfect", "ERA5"], 
    [graphcast_control_predictions.drop_vars(['initial_conditions_path']), graphcast_perfect_predictions.drop_vars(['initial_conditions_path']), era5_data]
):

    logger.info(f"Dataset: {dataset}...")

    daily_data = data.resample(time='1D').mean(dim='time')
    appended_data = xr.concat(
        [lookback_data, daily_data],
        dim='time'
    )

    VP200 = xr.zeros_like(appended_data['u_component_of_wind'].sel(level=200, drop=True))
    VP200[:] = vp(
        appended_data['u_component_of_wind'].sel(level=200).values,
        appended_data['v_component_of_wind'].sel(level=200).values
    )
    VP200.name = 'VP200'
    U200 = appended_data['u_component_of_wind'].sel(level=200, drop=True)
    U200.name = 'U200'
    U850 = appended_data['u_component_of_wind'].sel(level=850, drop=True)
    U850.name = 'U850'

    concatenated_data = {
        'VP200':VP200,
        'U200':U200,
        'U850':U850
    }

    VPM[dataset] = calculate_velocity_potential_indices_numpy(
        concatenated_data,
        init_date,
        VPM_datasets
    )

logger.info("Finished")

In [ ]:
projected_specific_humidity = xr.dot(
    graphcast_control_predictions['geopotential'],
    VPM['Control'],
    dim='time'
)

projected_specific_humidity.sel(mode=1, level=850).plot()
plt.figure()
projected_specific_humidity.sel(mode=2, level=850).plot()

In [ ]:
from auxiliary_functions.plotting_utils import bmh_colors
mode_to_plot = 1
time_to_plot = slice(init_date, np.datetime64(init_date)+np.timedelta64(10, 'D'))
plt.style.use('bmh')
plt.plot(
    reference_VPM.sel(mode=1, time=time_to_plot),
    reference_VPM.sel(mode=2, time=time_to_plot),
    color='gray',
    ls='--',
    label='Ventrice (2013)'
)
plt.plot(
    reference_VPM.sel(mode=1, time=time_to_plot.start),
    reference_VPM.sel(mode=2, time=time_to_plot.start),
    color='gray',
    marker='^',
    markersize=10
)
plt.plot(
    reference_VPM.sel(mode=1, time=time_to_plot.stop),
    reference_VPM.sel(mode=2, time=time_to_plot.stop),
    color='gray',
    marker='o',
    markersize=10
)
for index, data_source in enumerate(['Control', 'Perfect']):
    plt.plot(
        VPM[data_source].sel(mode=1, time=time_to_plot),
        VPM[data_source].sel(mode=2, time=time_to_plot),
        color=('k' if index == -1 else bmh_colors(index+1)),
        label=data_source
    )

    plt.plot(
        VPM[data_source].sel(mode=1, time=time_to_plot.start),
        VPM[data_source].sel(mode=2, time=time_to_plot.start),
        marker='^',
        markersize=10,
        color=('k' if index == -1 else bmh_colors(index+1)),
    )
    plt.plot(
        VPM[data_source].sel(mode=1, time=time_to_plot.stop),
        VPM[data_source].sel(mode=2, time=time_to_plot.stop),
        marker='o',
        markersize=10,
        color=('k' if index == -1 else bmh_colors(index+1)),
    )
# plt.plot(
#     VPM['Control'].sel(mode=1, time=time_to_plot),
#     VPM['Control'].sel(mode=2, time=time_to_plot),
#     label='Control'
# )
# plt.plot(
#     VPM['Perfect'].sel(mode=1, time=time_to_plot),
#     VPM['Perfect'].sel(mode=2, time=time_to_plot),
#     label='Perfect'
# )
plt.legend()

plt.axvline(0, color='k', ls=':', lw=1)
plt.axhline(0, color='k', ls=':', lw=1)

plt.title(f"Init Date: {np.datetime64(time_to_plot.start).astype('M8[ms]').astype('O').strftime('%HUTC %d-%^b-%Y')}", pad=5)
# plt.xlim(-1, 0.35)
# plt.ylim(-1.3, 0.1)
my_circle = plt.Circle((0, 0), radius=1, color='k', fill=False, linewidth=1)

# 3. Add the circle patch to the axes
plt.gca().add_patch(my_circle)
plt.gca().set_aspect('equal')
plt.show()

In [ ]:
# control_loss = np.sqrt(
#     (VPM['Control'].sel(mode = 1, drop=True) - VPM['ERA5'].sel(mode = 1, drop=True))**2
#     + (VPM['Control'].sel(mode = 1, drop=True) - VPM['ERA5'].sel(mode = 1, drop=True))**2
# )

# perfect_loss = np.sqrt(
#     (VPM['Perfect'].sel(mode = 1, drop=True) - VPM['ERA5'].sel(mode = 1, drop=True))**2
#     + (VPM['Perfect'].sel(mode = 1, drop=True) - VPM['ERA5'].sel(mode = 1, drop=True))**2
# )

control_loss = np.sqrt(
    (VPM['Control'].sel(mode = 1, drop=True) - reference_VPM.sel(time=VPM['Control'].time).sel(mode = 1, drop=True))**2
    + (VPM['Control'].sel(mode = 1, drop=True) - reference_VPM.sel(time=VPM['Control'].time).sel(mode = 1, drop=True))**2
)

perfect_loss = np.sqrt(
    (VPM['Perfect'].sel(mode = 1, drop=True) - reference_VPM.sel(time=VPM['Control'].time).sel(mode = 1, drop=True))**2
    + (VPM['Perfect'].sel(mode = 1, drop=True) - reference_VPM.sel(time=VPM['Control'].time).sel(mode = 1, drop=True))**2
)

control_loss.plot(label='Control')
perfect_loss.plot(label='Perfect')
plt.title("Loss")
plt.ylabel('RMSE')
plt.legend()
plt.show()

## Load data at multiple lead days

In [ ]:
# lead_days = np.arange(0, 3, 1)
lead_days = xr.DataArray(
    data = np.arange(0, 3, 1),
    dims='lead_day',
    coords={'lead_day':np.arange(0, 3, 1)},
    name='Lead Days'
)
init_date = "2020-12-28T00"
end_date = "2021-02-14"
run_notes = "_" + "ventrice_VPM"
forecast_initialization_dates = [np.datetime64(init_date) - np.timedelta64(lead_day, 'D') for lead_day in lead_days.values]
data_source = 'control'

logger.info("Loading ERA5 data...")
era5_initial_conditions_filepath = str(graphcast_control_predictions['initial_conditions_path'].values)
era5_data = xr.open_dataset(era5_initial_conditions_filepath, drop_variables=vars_to_drop+["land_sea_mask", "geopotential_at_surface", "toa_incident_solar_radiation"])
era5_data = era5_data.assign_coords(time=era5_data.datetime.isel(batch=0, drop=True)).drop_vars(['datetime'])
era5_data = era5_data.sel(
    time=slice(
        np.datetime64(init_date),
        np.datetime64(end_date)
    )
)

predicted_data = {}
for data_source in ['control', 'perfect']:
# --- control model forecast --
    logger.info(f"Loading Graphcast '{data_source.title()}' Predictions...")
    predicted_data[data_source.title()] = {}
    for lead_day, initialization_date in zip(lead_days.values, forecast_initialization_dates):
        if data_source == 'perfect' and lead_day == 0:
            continue

        logger.info(f"-- {initialization_date.astype('str')}, lead day(s): {lead_day}")

        prediction_timesteps = (
            (
                np.datetime64(end_date) - np.datetime64(initialization_date) + np.timedelta64(1, 'D')
                ).astype("timedelta64[h]")/6
            ).astype('int')

        prediction_filepath = (
            f"{config.GRAPHCAST_DATA_DIRECTORY}/{data_source}_model_forecasts/{initialization_date.astype('str')}/{initialization_date.astype('str')}_{prediction_timesteps}{run_notes}.zarr"
        )
        # init_date = prediction_filepath.split("/")[-1].split("_")[0]

        predictions = xr.open_zarr(prediction_filepath, drop_variables=vars_to_drop).load().sel(batch=0, drop=True)
        datetimes = convert_ns_to_datetime(predictions.time, initialization_date.astype('str'))
        predicted_data[data_source.title()][lead_day] = predictions.assign_coords({'time':datetimes.datetime}).drop_vars(["datetime"])

logger.info("Finished")

In [ ]:
concat_control_data = xr.concat(
    [
        predicted_data['Control'][lead_day].sel(time=slice(np.datetime64(init_date), None))
        for lead_day in lead_days.values
    ],
    dim=lead_days
)

concat_perfect_data = xr.concat(
    [
        predicted_data['Perfect'][lead_day].sel(time=slice(np.datetime64(init_date), None))
        for lead_day in lead_days.sel(lead_day=slice(1, None)).values
    ],
    dim=lead_days.sel(lead_day=slice(1, None))
)

In [ ]:
era5_data['u_component_of_wind'].sel(level=200, lat=slice(-15,15), lon=180).mean(dim='lat').plot(x='time', color='k')
for index, lead_day in enumerate(lead_days[1:]):
    concat_control_data['u_component_of_wind'].sel(
        lead_day=lead_day, level=200, lat=slice(-15,15), lon=180
    ).mean(dim='lat').plot(x='time', color=bmh_colors(index+1))
    concat_perfect_data['u_component_of_wind'].sel(
        lead_day=lead_day, level=200, lat=slice(-15,15), lon=180
    ).mean(dim='lat').plot(x='time', color=bmh_colors(index+1), ls=':')

In [ ]:
control_data_correlation = xr.corr(
    era5_data['u_component_of_wind'].sel(level=200, lat=slice(-15,15), lon=180).mean(dim='lat'),
    concat_control_data['u_component_of_wind'].sel(
            level=200, lat=slice(-15,15), lon=180
        ).mean(dim='lat'),
        dim='time'
)**2
perfect_data_correlation = xr.corr(
    era5_data['u_component_of_wind'].sel(level=200, lat=slice(-15,15), lon=180).mean(dim='lat'),
    concat_perfect_data['u_component_of_wind'].sel(
            level=200, lat=slice(-15,15), lon=180
        ).mean(dim='lat'),
        dim='time'
)**2

In [ ]:
concatenated_data = {}
VPM = {}

vp = VelocityPotentialNumpy(nlat=181, nlon=360, truncation=42)

import sys
sys.path.append("/glade/u/home/sressel/thesis-work/python/graphcast/gc-initial-condition-optimization/batch_modules")
from vpm_loss_utils import load_data_for_vpm 
VPM_datasets = load_data_for_vpm(init_date, prediction_timesteps)
lookback_data = VPM_datasets['Lookback Data']

for dataset, data in zip(
    ["Control", "Perfect", "ERA5"], 
    [
        concat_control_data.drop_vars(['initial_conditions_path']),
        concat_perfect_data.drop_vars(['initial_conditions_path']), 
        era5_data
        ]
):

    logger.info(f"Dataset: {dataset}...")

    daily_data = data.resample(time='1D').mean(dim='time')
    appended_data = xr.concat(
        [lookback_data, daily_data],
        dim='time'
    ).transpose('time', 'batch',  ..., 'lat', 'lon')

    VP200 = xr.zeros_like(appended_data['u_component_of_wind'].sel(level=200, drop=True))
    VP200[:] = vp(
        appended_data['u_component_of_wind'].sel(level=200).values,
        appended_data['v_component_of_wind'].sel(level=200).values
    )
    VP200.name = 'VP200'
    U200 = appended_data['u_component_of_wind'].sel(level=200, drop=True)
    U200.name = 'U200'
    U850 = appended_data['u_component_of_wind'].sel(level=850, drop=True)
    U850.name = 'U850'

    concatenated_data = {
        'VP200':VP200,
        'U200':U200,
        'U850':U850
    }

    VPM[dataset] = calculate_velocity_potential_indices_numpy(
        concatenated_data,
        init_date,
        VPM_datasets
    )

logger.info("Finished")

In [ ]:
fig = plt.figure(figsize=(9,9))
plt.plot(
    reference_VPM.sel(mode=1, time=slice(np.datetime64(init_date), np.datetime64(end_date))),
    reference_VPM.sel(mode=2, time=slice(np.datetime64(init_date), np.datetime64(end_date))),
    color='k'
)
plt.plot(
    reference_VPM.sel(mode=1, time=np.datetime64(init_date)),
    reference_VPM.sel(mode=2, time=np.datetime64(init_date)),
    color='k',
    marker='^',
    markersize=10
)
plt.plot(
    reference_VPM.sel(mode=1, time=np.datetime64(end_date)),
    reference_VPM.sel(mode=2, time=np.datetime64(end_date)),
    color='k',
    marker='o',
    markersize=10
)

for index, lead_day in enumerate(lead_days.values):
    plt.plot(
        VPM['Control'].sel(mode=1, lead_day=lead_day, time=slice(np.datetime64(init_date), np.datetime64(end_date))),
        VPM['Control'].sel(mode=2, lead_day=lead_day, time=slice(np.datetime64(init_date), np.datetime64(end_date))),
        color=bmh_colors(index+1)
    )
    plt.plot(
        VPM['Control'].sel(mode=1, lead_day=lead_day, time=np.datetime64(init_date)),
        VPM['Control'].sel(mode=2, lead_day=lead_day, time=np.datetime64(init_date)),
        color=bmh_colors(index+1),
        marker='^',
        markersize=10
    )
    plt.plot(
        VPM['Control'].sel(mode=1, lead_day=lead_day, time=np.datetime64(end_date)),
        VPM['Control'].sel(mode=2, lead_day=lead_day, time=np.datetime64(end_date)),
        color=bmh_colors(index+1),
        marker='o',
        markersize=10
    )
    if index != 0:
        plt.plot(
            VPM['Perfect'].sel(mode=1, lead_day=lead_day, time=slice(np.datetime64(init_date), np.datetime64(end_date))),
            VPM['Perfect'].sel(mode=2, lead_day=lead_day, time=slice(np.datetime64(init_date), np.datetime64(end_date))),
            color=bmh_colors(index+1),
            ls='--'
        )
        plt.plot(
            VPM['Perfect'].sel(mode=1, lead_day=lead_day, time=np.datetime64(init_date)),
            VPM['Perfect'].sel(mode=2, lead_day=lead_day, time=np.datetime64(init_date)),
            color=bmh_colors(index+1),
            marker='^',
            markersize=10,
            ls='--'
        )
        plt.plot(
            VPM['Perfect'].sel(mode=1, lead_day=lead_day, time=np.datetime64(end_date)),
            VPM['Perfect'].sel(mode=2, lead_day=lead_day, time=np.datetime64(end_date)),
            color=bmh_colors(index+1),
            marker='o',
            markersize=10,
            ls='--'
        )

circle1 = plt.Circle((0, 0), 1.0, color="k", fill=False, lw=1, zorder=10)
plt.hlines(y=0, xmin=-4, xmax=-1, color="k", lw=1, ls="-")
plt.hlines(y=0, xmin=1, xmax=4, color="k", lw=1, ls="-")
plt.vlines(x=0, ymin=-4, ymax=-1, color="k", lw=1, ls="-")
plt.vlines(x=0, ymin=1, ymax=4, color="k", lw=1, ls="-")
plt.plot([np.sqrt(2) / 2, 4], [np.sqrt(2) / 2, 4], color="k", lw=1, ls="-")
# Add lines to differentiate the phases
plt.plot([np.sqrt(2) / 2, 4], [-np.sqrt(2) / 2, -4], color="k", lw=1, ls="-")
plt.plot([-4, -np.sqrt(2) / 2], [4, np.sqrt(2) / 2], color="k", lw=1, ls="-")
plt.plot([-4, -np.sqrt(2) / 2], [-4, -np.sqrt(2) / 2], color="k", lw=1, ls="-")
plt.gca().add_patch(circle1)
plt.gca().set_aspect('equal')

# plt.xlim(-1, 1)
# plt.ylim(-2, 2)

plt.show()

In [ ]:
reference_amplitude = np.sqrt(reference_VPM.sel(mode=1)**2 + reference_VPM.sel(mode=2)**2)
control_amplitude = np.sqrt(VPM['Control'].sel(mode=1)**2 + VPM['Control'].sel(mode=2)**2)
perfect_amplitude = np.sqrt(VPM['Perfect'].sel(mode=1)**2 + VPM['Perfect'].sel(mode=2)**2)

In [ ]:
reference_amplitude.sel(time=slice(np.datetime64(init_date), np.datetime64(end_date))).plot(color='k')
control_amplitude.sel(time=slice(np.datetime64(init_date), np.datetime64(end_date)), lead_day=0).plot()
control_amplitude.sel(time=slice(np.datetime64(init_date), np.datetime64(end_date)), lead_day=1).plot()
control_amplitude.sel(time=slice(np.datetime64(init_date), np.datetime64(end_date)), lead_day=2).plot()
perfect_amplitude.sel(time=slice(np.datetime64(init_date), np.datetime64(end_date)), lead_day=1).plot(color=bmh_colors(2), ls='--')
perfect_amplitude.sel(time=slice(np.datetime64(init_date), np.datetime64(end_date)), lead_day=2).plot(color=bmh_colors(3), ls='--')

In [ ]:
loss_difference = np.sqrt(
    (VPM['Perfect'].sel(mode = 1, drop=True) - VPM['Control'].sel(mode = 1, drop=True))**2
    + (VPM['Perfect'].sel(mode = 1, drop=True) - VPM['Control'].sel(mode = 1, drop=True))**2
)

In [ ]:
loss_difference.sum('time').plot()

In [ ]:
xr.corr(
    reference_amplitude,
    control_amplitude,
    dim='time'
).print()
xr.corr(
    reference_amplitude,
    perfect_amplitude,
    dim='time'
).print()

## Plot horizontal structures

In [ ]:
# time_to_plot = "2003-11-19T00"
# time_to_plot = "2020-10-01T00"
time_to_plot = "2020-12-28T00"
variable_to_plot = 'u_component_of_wind'
level_to_plot = 200

SAVE_FIG = True
PLOT_MODE = "slides"
FIG_LAYOUT = 'full'
plt.style.use('bmh')
set_plot_mode(PLOT_MODE)
plt.rcParams['mathtext.fontset'] = 'dejavusans'
plt.rcParams['font.size'] = '12'
plt.rcParams['axes.titlesize'] = '12'
plt.rcParams['axes.labelsize'] = '12'
plt.rcParams['xtick.labelsize'] = '10'
plt.rcParams['ytick.labelsize'] = '10'
plt.rcParams['lines.linewidth'] = '1'

fig = plt.figure(figsize=get_figsize(PLOT_MODE, FIG_LAYOUT))

gs = GridSpec(4, 2, figure=fig, height_ratios=[100, 100, 100, 8])
gs.update(right=0.8, wspace=0.05, hspace=0.5)

axes = [
    fig.add_subplot(gs[0,0], projection=ccrs.PlateCarree(central_longitude=-180)),
    fig.add_subplot(gs[1,0], projection=ccrs.PlateCarree(central_longitude=-180)),
    fig.add_subplot(gs[2,0], projection=ccrs.PlateCarree(central_longitude=-180)),
    fig.add_subplot(gs[1,1], projection=ccrs.PlateCarree(central_longitude=-180)),
    fig.add_subplot(gs[2,1], projection=ccrs.PlateCarree(central_longitude=-180)),
]
cbar_axes = [
    fig.add_subplot(gs[-1, 0]),
    fig.add_subplot(gs[-1, 1])
]
data_crs = ccrs.PlateCarree()
# levels = np.linspace(0, 15, 11)/1000
# levels = np.linspace(0, 0.01, 17)

axes[0].set_title("a) ERA5 Data", loc='left')
im = axes[0].contourf(
    era5_data['lon'],
    era5_data['lat'],
    era5_data[variable_to_plot].coord_funcs.sel(time=time_to_plot, level=level_to_plot, batch=0),
    transform=data_crs,
    levels=17,
    cmap='viridis',
    extend='max'
)
cbar = fig.colorbar(im, cax=cbar_axes[0], orientation='horizontal')
cbar.set_label(f"{config.VARIABLE_UNITS[variable_to_plot]}")
cbar.set_ticks(im.levels[::4])

axes[1].set_title("b) 'Control' Prediction", loc='left')
axes[1].contourf(
    graphcast_control_predictions['lon'],
    graphcast_control_predictions['lat'],
    graphcast_control_predictions[variable_to_plot].coord_funcs.sel(time=time_to_plot, level=level_to_plot, batch=0),
    transform=data_crs,
    levels=im.levels,
    cmap='viridis',
    extend='max'
)

axes[2].set_title("c) 'Perfect' Prediction", loc='left')
axes[2].contourf(
    graphcast_perfect_predictions['lon'],
    graphcast_perfect_predictions['lat'],
    graphcast_perfect_predictions[variable_to_plot].coord_funcs.sel(time=time_to_plot, level=level_to_plot, batch=0),
    transform=data_crs,
    levels=im.levels,
    cmap='viridis',
    extend='max'
)

axes[3].set_title("d) ERA5 - 'Control'", loc='left')
im_diff = axes[3].contourf(
    graphcast_perfect_predictions['lon'],
    graphcast_perfect_predictions['lat'],
    (
        era5_data[variable_to_plot].coord_funcs.sel(time=time_to_plot, level=level_to_plot, batch=0)
        - graphcast_control_predictions[variable_to_plot].coord_funcs.sel(time=time_to_plot, level=level_to_plot, batch=0)
        ),
    transform=data_crs,
    # levels=np.linspace(0, 15, 11),
    levels=17,
    cmap='coolwarm',
    norm=mcolors.CenteredNorm(vcenter=0),
    extend='both'
)
cbar = fig.colorbar(im_diff, cax=cbar_axes[1], orientation='horizontal')
cbar.set_label(f"{config.VARIABLE_UNITS[variable_to_plot]}")
cbar.set_ticks(im_diff.levels[::4])

axes[4].set_title("e) ERA5 - 'Perfect'", loc='left')
axes[4].contourf(
    graphcast_perfect_predictions['lon'],
    graphcast_perfect_predictions['lat'],
    (
        era5_data[variable_to_plot].coord_funcs.sel(time=time_to_plot, level=level_to_plot, batch=0)
        - graphcast_perfect_predictions[variable_to_plot].coord_funcs.sel(time=time_to_plot, level=level_to_plot, batch=0)
        ),
    transform=data_crs,
    # levels=np.linspace(0, 15, 11),
    levels=im_diff.levels,
    cmap='coolwarm',
    norm=mcolors.CenteredNorm(vcenter=0),
    extend='both'
)
for axis in axes:
    axis.add_feature(cfeature.COASTLINE, linewidth=0.5)
    axis.set_aspect('equal')
    # gl = axis.gridlines(crs=data_crs, draw_labels=True,
    #                 linewidth=1, color='#bcbcbc', alpha=0.5, linestyle='-')
    # gl.right_labels = False
    # gl.top_labels = False # Hides x-labels on the bottom
    # gl.xlocator = mticker.FixedLocator([60, 90, 120, 150, 180])
    # gl.ylocator = mticker.FixedLocator([-30, -15, 0, 15, 30])
    # gl.xformatter = LongitudeFormatter()
    # gl.yformatter = LatitudeFormatter()
    axis.set_extent([60, 180, -30, 30], crs=data_crs)

fig.suptitle(
    (
        f"{np.datetime64(time_to_plot).astype('M8[ms]').astype('O').strftime('%HUTC %d-%^b-%Y')}"
        + f"\n{config.VARIABLE_SHORTNAMES[variable_to_plot]}" 
        + (f"{level_to_plot}" if 'level' in era5_data[variable_to_plot].coords else "")
    ),
    x=0.5, y=1.01, ha="center"
)

# Save figure
output_filename = (
    f"{np.datetime64(time_to_plot).astype('M8[ms]').astype('O').strftime('%Y-%m-%dT%H')}"
    + f"_{config.VARIABLE_SHORTNAMES[variable_to_plot]}"
    + (f"{level_to_plot}" if 'level' in era5_data[variable_to_plot].coords else "")
)
logger.info(f"Output directory: {config.OUTPUT_DIRECTORY}/era5_control_perfect_comparisons/")
logger.info(f"Output filename: {output_filename}")
if SAVE_FIG:
    logger.info(f"Saving...")
    plt.savefig(
        f"{config.OUTPUT_DIRECTORY}/{output_filename}.svg",
        dpi=500,
        bbox_inches="tight",
    )
else:
    logger.info("Not Saving")

logger.info("Finished")
plt.show()

In [ ]:
# time_to_plot = "2021-01-06T00"
time_to_plot = "2020-12-14T00"
level_to_plot = 850
variable_to_plot = 'specific_humidity'
# unit_multiplier = 1000
# units_label = r"g kg$^{-1}$"

# variable_to_plot = 'total_precipitation_6hr'
# unit_multiplier = 1000/6
# units = r"mm hr$^{-1}$"

SAVE_FIG = True
PLOT_MODE = "slides"
FIG_LAYOUT = 'full'
plt.style.use('bmh')
set_plot_mode(PLOT_MODE)

plt.rcParams['mathtext.fontset'] = 'dejavusans'
plt.rcParams['font.size'] = '12'
plt.rcParams['axes.titlesize'] = '12'
plt.rcParams['axes.labelsize'] = '12'
plt.rcParams['xtick.labelsize'] = '10'
plt.rcParams['ytick.labelsize'] = '10'
plt.rcParams['lines.linewidth'] = '1'

fig = plt.figure(figsize=get_figsize(PLOT_MODE, FIG_LAYOUT))

# ─────────────────────────────────────────────
# 2 rows × 4 columns
# Columns: [plot, plot, plot, colorbar]
# Row 1: ERA5, Control, Perfect
# Row 2: Control Diff, Perfect Diff
# ─────────────────────────────────────────────
gs = GridSpec(
    2, 4, figure=fig,
    width_ratios=[1, 1, 1, 0.08],
    wspace=0.35, hspace=0.35
)
gs.update(top=1)


# Row 1 axes
ax_era   = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree(central_longitude=-180))
ax_def   = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree(central_longitude=-180))
ax_perf  = fig.add_subplot(gs[0, 2], projection=ccrs.PlateCarree(central_longitude=-180))

# Row 2 axes
ax_diff_def  = fig.add_subplot(gs[1, 1], projection=ccrs.PlateCarree(central_longitude=-180))
ax_diff_perf = fig.add_subplot(gs[1, 2], projection=ccrs.PlateCarree(central_longitude=-180))

# Colorbar axes
cax_top = fig.add_subplot(gs[0, 3])
cax_bot = fig.add_subplot(gs[1, 3])

data_crs = ccrs.PlateCarree()

levels = np.linspace(0, 15, 11)/1000

# ─────────────────────────────────────────────
# Row 1: ERA5
# ─────────────────────────────────────────────
ax_era.set_title("a) ERA5 Data", loc='left')
im_top = ax_era.contourf(
    era5_data['lon'],
    era5_data['lat'],
    era5_data[variable_to_plot].coord_funcs.sel(time=time_to_plot, level=level_to_plot, batch=0), 
    transform=data_crs,
    levels=levels,
    cmap='viridis',
    extend='max'
)

# Control prediction
ax_def.set_title("b) 'Control' Prediction", loc='left')
ax_def.contourf(
    graphcast_control_predictions['lon'],
    graphcast_control_predictions['lat'],
    graphcast_control_predictions[variable_to_plot].coord_funcs.sel(time=time_to_plot, level=level_to_plot, batch=0), 
    transform=data_crs,
    levels=levels,
    cmap='viridis',
    extend='max'
)

# Perfect prediction
ax_perf.set_title("c) 'Perfect' Prediction", loc='left')
ax_perf.contourf(
    graphcast_perfect_predictions['lon'],
    graphcast_perfect_predictions['lat'],
    graphcast_perfect_predictions[variable_to_plot].coord_funcs.sel(time=time_to_plot, level=level_to_plot, batch=0), 
    transform=data_crs,
    levels=levels,
    cmap='viridis',
    extend='max'
)

# Colorbar for row 1
cbar = fig.colorbar(im_top, cax=cax_top)
cbar.set_label(f"{config.VARIABLE_UNITS[variable_to_plot]}")

# ─────────────────────────────────────────────
# Row 2: Differences
# ─────────────────────────────────────────────
ax_diff_def.set_title("d) ERA5 - 'Control'", loc='left')
im_diff = ax_diff_def.contourf(
    graphcast_perfect_predictions['lon'],
    graphcast_perfect_predictions['lat'],
    (
        era5_data[variable_to_plot].coord_funcs.sel(time=time_to_plot, level=level_to_plot, batch=0)
        - graphcast_control_predictions[variable_to_plot].coord_funcs.sel(time=time_to_plot, level=level_to_plot, batch=0)
    ),
    transform=data_crs,
    levels=21,
    cmap='coolwarm',
    norm=mcolors.CenteredNorm(vcenter=0),
    extend='both'
)

ax_diff_perf.set_title("e) ERA5 - 'Perfect'", loc='left')
ax_diff_perf.contourf(
    graphcast_perfect_predictions['lon'],
    graphcast_perfect_predictions['lat'],
    (
        era5_data[variable_to_plot].coord_funcs.sel(time=time_to_plot, level=level_to_plot, batch=0)
        - graphcast_perfect_predictions[variable_to_plot].coord_funcs.sel(time=time_to_plot, level=level_to_plot, batch=0)
    ),
    transform=data_crs,
    levels=im_diff.levels,
    cmap='coolwarm',
    norm=mcolors.CenteredNorm(vcenter=0),
    extend='both'
)

# Colorbar for row 2
cbar = fig.colorbar(im_diff, cax=cax_bot)
cbar.set_label(f"{config.VARIABLE_UNITS[variable_to_plot]}")

# ─────────────────────────────────────────────
# Shared map formatting
# ─────────────────────────────────────────────
all_axes = [ax_era, ax_def, ax_perf, ax_diff_def, ax_diff_perf]

for ax in all_axes:
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.set_aspect('equal')

    gl = ax.gridlines(
        crs=data_crs, draw_labels=True,
        linewidth=1, color='#bcbcbc', alpha=0.5, linestyle='-',
        x_inline=False,
    )
    gl.right_labels = False
    gl.top_labels = False
    gl.bottom_labels = True
    gl.xlocator = mticker.FixedLocator([60, 90, 120, 150, 180])
    gl.ylocator = mticker.FixedLocator([-30, -15, 0, 15, 30])
    gl.xformatter = LongitudeFormatter()
    gl.yformatter = LatitudeFormatter()

    ax.set_extent([60, 180, -30, 30], crs=data_crs)

# ─────────────────────────────────────────────
# Title
# ─────────────────────────────────────────────
fig.suptitle(
    (
        f"{np.datetime64(time_to_plot).astype('M8[ms]').astype('O').strftime('%HZ %d-%^b-%Y')}"
        + f" {config.VARIABLE_SHORTNAMES[variable_to_plot]}" 
        + (f"{level_to_plot}" if 'level' in era5_data[variable_to_plot].coords else "")
    ),
    x=0.5, y=1.01, ha="center"
)

# Save figure
output_filename = (
    f"{np.datetime64(time_to_plot).astype('M8[ms]').astype('O').strftime('%Y-%m-%dT%H')}"
    + f"_{config.VARIABLE_SHORTNAMES[variable_to_plot]}"
    + (f"{level_to_plot}" if 'level' in era5_data[variable_to_plot].coords else "")
)
logger.info(f"Output directory: {config.OUTPUT_DIRECTORY}/era5_control_perfect_comparisons/")
logger.info(f"Output filename: {output_filename}")
if SAVE_FIG:
    logger.info(f"Saving...")
    plt.savefig(
        f"{config.OUTPUT_DIRECTORY}/{output_filename}.svg",
        dpi=500,
        bbox_inches="tight",
    )
else:
    logger.info("Not Saving")

logger.info("Finished")
plt.show()

## Plot Hovmoller

In [ ]:
input_data_subset = era5_data.sel(
    datetime=slice(
        graphcast_control_predictions.time[0],
        graphcast_control_predictions.time[-1]
    )
)

variable_to_plot = 'total_precipitation_6hr'
level_to_plot = 1000
# unit_multiplier = (1000)
# unit_multiplier = 1

SAVE_FIG = True
PLOT_MODE = "slides"
FIG_LAYOUT = 'full'
plt.style.use('bmh')
set_plot_mode(PLOT_MODE)
plt.rcParams['mathtext.fontset'] = 'dejavusans'
plt.rcParams['font.size'] = '14'
plt.rcParams['axes.labelsize'] = '14'
plt.rcParams['xtick.labelsize'] = '12'
plt.rcParams['ytick.labelsize'] = '12'
plt.rcParams['lines.linewidth'] = '2'

fig = plt.figure(figsize=get_figsize(PLOT_MODE, FIG_LAYOUT))

gs = GridSpec(2, 3, figure=fig, height_ratios=[30,1])
gs.update(top=1, bottom=0, left=0, right=1, wspace=0.1)

axes = [
    fig.add_subplot(gs[0,0]),
    fig.add_subplot(gs[0,1]),
    fig.add_subplot(gs[0,2]),
]
cbar_axes = [
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1:]),
]

axes[0].set_title('ERA5 Data')
im_era5 = axes[0].contourf(
    input_data_subset[variable_to_plot].lon,
    input_data_subset[variable_to_plot].datetime,
    (input_data_subset[variable_to_plot].coord_funcs.sel(lat=slice(-10,10), level=level_to_plot).mean(dim='lat')),
    # levels=np.arange(0, 2.25, 0.25)
    levels=21,
)
cbar = fig.colorbar(im_era5, cax=cbar_axes[0], orientation='horizontal')
cbar.set_label(f"{config.VARIABLE_UNITS[variable_to_plot]}")
cbar.set_ticks(im_era5.levels[::8])

axes[1].set_title("ERA5 - 'Control' Prediction")
im_control = axes[1].contourf(
    graphcast_control_predictions.lon,
    graphcast_control_predictions.datetime,
    ((input_data_subset[variable_to_plot].coord_funcs.sel(lat=slice(-10,10), level=level_to_plot).mean(dim='lat'))
     - (graphcast_control_predictions[variable_to_plot].coord_funcs.sel(lat=slice(-10,10), level=level_to_plot).mean(dim='lat'))
    ),
    levels=17,#im_era5.levels,
    cmap='coolwarm',
    norm=mcolors.CenteredNorm(vcenter=0)
)

axes[2].set_title("ERA5 - 'Perfect' Prediction")
im_perfect = axes[2].contourf(
    graphcast_perfect_predictions.lon,
    graphcast_perfect_predictions.datetime,
    ((input_data_subset[variable_to_plot].coord_funcs.sel(lat=slice(-10,10), level=level_to_plot).mean(dim='lat'))
     - (graphcast_perfect_predictions[variable_to_plot].coord_funcs.sel(lat=slice(-10,10), level=level_to_plot).mean(dim='lat'))
    ),
    levels=im_control.levels,#im_era5.levels,
    cmap='coolwarm',
    norm=mcolors.CenteredNorm(vcenter=0)
)
cbar = fig.colorbar(im_control, cax=cbar_axes[1], orientation='horizontal')
cbar.set_label(f"{config.VARIABLE_UNITS[variable_to_plot]}")
cbar.set_ticks(im_control.levels[::4])


for index, ax in enumerate(axes):
    ax.set_yticks(input_data_subset.datetime[::4])
    if index != 0:
        ax.set_yticklabels('')
    ax.set_xticks(np.arange(0, 360, 60), labels=tick_labeller(np.arange(0, 360, 60), 'lon'))
    ax.grid(axis='y', alpha=0.5)

fig.suptitle(
    (
        f"{config.VARIABLE_SHORTNAMES[variable_to_plot]}" 
        + (f"{level_to_plot}" if 'level' in era5_data[variable_to_plot].coords else "")
    ),
    x=0.5, y=1.10, ha="center"
)


# Save figure
output_filename = (
    f"{config.VARIABLE_SHORTNAMES[variable_to_plot]}"
    + (f"{level_to_plot}" if 'level' in era5_data[variable_to_plot].coords else "")
    + "_hovmoller"
)
logger.info(f"Output directory: {config.OUTPUT_DIRECTORY}/era5_control_perfect_comparisons/")
logger.info(f"Output filename: {output_filename}")
if SAVE_FIG:
    logger.info(f"Saving...")
    plt.savefig(
        f"{config.OUTPUT_DIRECTORY}/{output_filename}.svg",
        dpi=500,
        bbox_inches="tight",
    )
else:
    logger.info("Not Saving")

logger.info("Finished")
plt.show()

In [ ]:
(graphcast_control_predictions['specific_humidity'] - graphcast_perfect_predictions['specific_humidity']).sel(lat=slice(-15,15), level=850).mean(dim='lat').plot.contourf(x='lon', y='datetime', levels=21, norm=mcolors.CenteredNorm(vcenter=0), extend='both')

## Plot horizontal structures

In [ ]:
# variable_to_plot = 'total_precipitation_6hr'
# unit_multiplier = (1000/6)
# variable_units = r"mm hr$^{-1}$"

variable_to_plot = 'specific_humidity'
unit_multiplier = (1000)
variable_units = r"g kg$^{-1}$"
plev_to_plot = 850

time_to_plot = np.datetime64(init_date) + np.timedelta64(10, 'D')


SAVE_FIG = False
PLOT_MODE = "slides"
FIG_LAYOUT = 'full'
plt.style.use('bmh')
set_plot_mode(PLOT_MODE)


fig = plt.figure(figsize=get_figsize(PLOT_MODE, FIG_LAYOUT))
gs = GridSpec(3, 2, figure=fig, width_ratios=[30, 1])
gs.update(top=1, bottom=0, left=0, right=1, hspace=0.35, wspace=0.1)

central_longitude = 180
proj = ccrs.PlateCarree(central_longitude=central_longitude)
data_crs = ccrs.PlateCarree()
coastline_width = 1


ax = [
    fig.add_subplot(gs[0,0], projection=proj),
    fig.add_subplot(gs[1,0], projection=proj),
    fig.add_subplot(gs[2,0], projection=proj),
]
cbar_ax = [
    fig.add_subplot(gs[:2,-1]),
    fig.add_subplot(gs[2,-1])
]

cdata_input = xarray_utils.add_cyclic_point(
    era5_data[variable_to_plot].coord_funcs.sel(level=850),
    dim='lon'
)

cdata_pred = xarray_utils.add_cyclic_point(
    graphcast_perfect_predictions[variable_to_plot].coord_funcs.sel(level=850),
    dim='lon'
)

ax[0].set_title("a) ERA5", loc='left')
im = ax[0].contourf(
    cdata_input.lon,
    cdata_input.lat,
    (unit_multiplier*cdata_input).sel(datetime=time_to_plot),
    levels=np.arange(0, 27.5, 2.5), 
    # levels=np.linspace(-21, 21, 21),
    transform=data_crs,
    cmap='viridis', 
    extend='max'
)

ax[1].set_title("b) Graphcast", loc='left')
ax[1].contourf(
    cdata_pred.lon,
    cdata_pred.lat,
    (unit_multiplier*cdata_pred).sel(datetime=time_to_plot),
    # levels=np.arange(0, 27.5, 2.5), 
    levels=im.levels, 
    transform=data_crs,
    cmap='viridis', 
    extend='max'
)
# Add colorbar
cbar = fig.colorbar(im, cax=cbar_ax[0])
cbar.set_label(f"{variable_units}")

ax[2].set_title("c) ERA5 - Graphcast", loc='left')
im_diff = ax[2].contourf(
    cdata_pred.lon,
    cdata_pred.lat,
    (
        (unit_multiplier*cdata_pred).sel(datetime=time_to_plot)
         - (unit_multiplier*cdata_input).sel(datetime=time_to_plot)
    ),
    levels=np.linspace(-16, 15, 11), 
    transform=data_crs,
    cmap='coolwarm', 
    norm=mcolors.CenteredNorm(vcenter=0),
    extend='both'
)
cbar = fig.colorbar(im_diff, cax=cbar_ax[1])
cbar.set_label(f"{variable_units}")

for axis in ax:
    # axis.set_aspect('equal')
    axis.add_feature(cfeature.COASTLINE, lw=coastline_width)
    minlon = central_longitude - 120
    maxlon = central_longitude
    axis.set_extent([minlon, maxlon, -30, 30], data_crs)
    gl = axis.gridlines(
        crs=data_crs,
        draw_labels=True,
        linewidth=1,
        color="gray",
        alpha=0.75,
        linestyle="-",
        zorder=15
    )
    gl.right_labels = False
    gl.top_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(60,210,15))
    gl.xformatter = LongitudeFormatter()
    gl.ylocator = mticker.FixedLocator(np.arange(-30,45,15))
    gl.yformatter = LatitudeFormatter()

fig.suptitle(
    f"{np.datetime64(time_to_plot).astype('M8[ms]').astype('O').strftime('%d-%^b-%Y')}",
    x=ax[0].get_position().x0 + 0.5 * ax[0].get_position().width,
    y=1.05,
    ha="center"
)

output_filename = f"ERA5 vs. Graphcast Horizontal Structures - {np.datetime64(time_to_plot).astype('M8[ms]').astype('O').strftime('%Y-%m-%d-T%H')}.svg"
logger.info(f"Output directory: {config.OUTPUT_DIRECTORY}/")
logger.info(f"Output filename: {output_filename}")
if SAVE_FIG:
    logger.info("Saving...")
    plt.savefig(
        f"{config.OUTPUT_DIRECTORY}/{output_filename}.svg",
        dpi=500,
        bbox_inches="tight",
    )
else:
    logger.info("Not Saving")
logger.info("Finished")

plt.show()

In [ ]:
variable_to_plot = 'total_precipitation_6hr'
plev_to_plot = 850

times_to_plot = np.arange(
    np.datetime64(init_date) + np.timedelta64(1, 'D'),
    np.datetime64(init_date) + np.timedelta64(21, 'D'),
    np.timedelta64(2, 'D')
    )

fig = plt.figure(figsize=(12, len(times_to_plot)))
gs = GridSpec(len(times_to_plot), 2, figure=fig, width_ratios=[30, 1])
gs.update(top=1, bottom=0, left=0, right=0.8, hspace=0.1, wspace=0.05)

proj = ccrs.PlateCarree(central_longitude=180)
data_crs = ccrs.PlateCarree()

ax = [
    fig.add_subplot(gs[i, 0], projection=proj) 
    for i in range(len(times_to_plot))
]

cbar_axis = fig.add_subplot(gs[:, 1])

cf_data = xarray_utils.add_cyclic_point(
    predicted_data[variable_to_plot].coord_funcs.sel(level=plev_to_plot),
    dim='lon'
).stats.standardize(dim=['lat', 'lon'])

quiv_u_data = xarray_utils.add_cyclic_point(
    predicted_data['u_component_of_wind'].coord_funcs.sel(level=plev_to_plot),
    dim='lon'
)

quiv_v_data = xarray_utils.add_cyclic_point(
    predicted_data['v_component_of_wind'].coord_funcs.sel(level=plev_to_plot),
    dim='lon'
)

# ax[0].set_title(f"Event starting {start_time.astype('M8[ms]').astype('O').strftime('%d%^b%Y')}", fontsize=16)
for index, time in enumerate(times_to_plot):
    im = ax[index].contourf(
        cf_data.lon,
        cf_data.lat,
        cf_data.sel(datetime=time),
        # levels=np.arange(-125, 125, 25),
        levels=21,
        transform=data_crs,
        cmap='coolwarm',
        norm=mcolors.CenteredNorm(vcenter=0),
    )
    ax[index].add_feature(cfeature.COASTLINE, lw=1)
    # ax[index].text(
    #     s=f"{(time-primary_event_max_times[event]).values.astype('timedelta64[D]')}", 
    #     x=1-0.015, 
    #     y=0.9,
    #     transform=ax[index].transAxes,
    #     fontsize=12,
    #     verticalalignment='top',
    #     horizontalalignment='right',
    #     bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='none')
    #     )

    arrow_spacing = 4
    ax[index].quiver(
        quiv_u_data.lon[::2*arrow_spacing],
        quiv_v_data.lat[::arrow_spacing],
        quiv_u_data.sel(datetime=time)[::arrow_spacing, ::2*arrow_spacing].values,
        quiv_v_data.sel(datetime=time)[::arrow_spacing, ::2*arrow_spacing].values,
        transform=data_crs,
        width=0.002,
        # scale=200
    )

    minlon = central_longitude - 180
    maxlon = central_longitude + 180
    ax[index].set_extent([minlon, maxlon, -30, 30], data_crs)
    gl = ax[index].gridlines(
        crs=data_crs,
        draw_labels=(True if index == len(times_to_plot) - 1 else False),
        linewidth=1,
        color="gray",
        alpha=0.75,
        linestyle="-",
        zorder=15

    )
    gl.right_labels = False
    gl.top_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(-180,180,60))
    gl.xformatter = LongitudeFormatter()
    gl.ylocator = mticker.FixedLocator(np.arange(-30,45,15))
    gl.yformatter = LatitudeFormatter()

fig.colorbar(im, cax=cbar_axis, orientation='vertical')

plt.show()

In [ ]:
variable_to_plot = 'specific_humidity'
plev_to_plot = 850

times_to_plot = np.arange(
    np.datetime64(init_date) + np.timedelta64(1, 'D'),
    np.datetime64(init_date) + np.timedelta64(30, 'D'),
    np.timedelta64(4, 'D')
)

fig = plt.figure(figsize=(12, len(times_to_plot)))
gs = GridSpec(len(times_to_plot), 2, figure=fig, width_ratios=[30, 1])
gs.update(top=1, bottom=0, left=0, right=0.8, hspace=0.1, wspace=0.05)

proj = ccrs.PlateCarree(central_longitude=180)
data_crs = ccrs.PlateCarree()

ax = [
    fig.add_subplot(gs[i, 0], projection=proj) 
    for i in range(len(times_to_plot))
]

cbar_axis = fig.add_subplot(gs[:, 1])

cf_data = xarray_utils.add_cyclic_point(
    era5_data[variable_to_plot].coord_funcs.sel(level=plev_to_plot),
    dim='lon'
).sel(lat=slice(-30,30)).stats.standardize(dim=['lat', 'lon'])

quiv_u_data = xarray_utils.add_cyclic_point(
    era5_data['u_component_of_wind'].coord_funcs.sel(level=plev_to_plot),
    dim='lon'
).sel(lat=slice(-30,30))

quiv_v_data = xarray_utils.add_cyclic_point(
    era5_data['v_component_of_wind'].coord_funcs.sel(level=plev_to_plot),
    dim='lon'
).sel(lat=slice(-30,30))

ax[0].set_title(f"Event starting {np.datetime64(init_date).astype('M8[ms]').astype('O').strftime('%d%^b%Y')}", fontsize=16)
for index, time in enumerate(times_to_plot):
    im = ax[index].contourf(
        cf_data.lon,
        cf_data.lat,
        cf_data.sel(datetime=time),
        # levels=np.arange(-125, 125, 25),
        levels=21,
        transform=data_crs,
        cmap='coolwarm',
        norm=mcolors.CenteredNorm(vcenter=0),
    )
    ax[index].add_feature(cfeature.COASTLINE, lw=1)
    ax[index].text(
        s=f"{(time-times_to_plot[0]).astype('timedelta64[D]')}", 
        x=1-0.015, 
        y=0.9,
        transform=ax[index].transAxes,
        fontsize=12,
        verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='none')
        )

    arrow_spacing = 8
    ax[index].quiver(
        quiv_u_data.lon[::2*arrow_spacing],
        quiv_v_data.lat[::arrow_spacing],
        quiv_u_data.sel(datetime=time)[::arrow_spacing, ::2*arrow_spacing].values,
        quiv_v_data.sel(datetime=time)[::arrow_spacing, ::2*arrow_spacing].values,
        transform=data_crs,
        width=0.002,
        # scale=200
    )

    minlon = central_longitude - 180
    maxlon = central_longitude + 180
    ax[index].set_extent([minlon, maxlon, -30, 30], data_crs)
    gl = ax[index].gridlines(
        crs=data_crs,
        draw_labels=(True if index == len(times_to_plot) - 1 else False),
        linewidth=1,
        color="gray",
        alpha=0.75,
        linestyle="-",
        zorder=15

    )
    gl.right_labels = False
    gl.top_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(-180,180,60))
    gl.xformatter = LongitudeFormatter()
    gl.ylocator = mticker.FixedLocator(np.arange(-30,45,15))
    gl.yformatter = LatitudeFormatter()

fig.colorbar(im, cax=cbar_axis, orientation='vertical')

plt.show()

# Recreate Vonich & Hakim (2021)

In [ ]:
logger.info("Load Graphcast Prediction Data")
prediction_initial_date = "2021-06-20T00"
year_directory = "202105_202106"
# prediction_initial_date = "2021-01-01T00"
# year_directory = "202012_202102"
prediction_timesteps = 40
run_notes = ""
logger.info(f"Initial Date: {prediction_initial_date}, {prediction_timesteps//config.TIMESTEPS_PER_DAY} day prediction, {run_notes}")

logger.info("Loading control predicted data...")
control_prediction_filepath = f"{config.GRAPHCAST_DATA_DIRECTORY}/control_model_forecasts/{prediction_initial_date}/{prediction_initial_date}_{prediction_timesteps}{run_notes}.zarr"
init_date = control_prediction_filepath.split("/")[-1].split("_")[0]

graphcast_control_predictions = xr.open_zarr(control_prediction_filepath).load().sel(batch=0, drop=True).drop_vars(['number', 'expver'])
datetimes = convert_ns_to_datetime(graphcast_control_predictions.time, init_date)
graphcast_control_predictions = graphcast_control_predictions.assign_coords({'datetime':datetimes.datetime}).swap_dims({'time':'datetime'})

logger.info("Loading global perfect predicted data...")
perfect_prediction_filepath_global = f"{config.GRAPHCAST_DATA_DIRECTORY}/perfect_model_forecasts/{prediction_initial_date}/{prediction_initial_date}_{prediction_timesteps}_global.zarr"
perfect_predicted_data_global = xr.open_zarr(perfect_prediction_filepath_global).load().sel(batch=0, drop=True).drop_vars(['number', 'expver'])
datetimes = convert_ns_to_datetime(perfect_predicted_data_global.time, init_date)
perfect_predicted_data_global = perfect_predicted_data_global.assign_coords({'datetime':datetimes.datetime}).swap_dims({'time':'datetime'})

logger.info("Loading subset perfect predicted data...")
perfect_prediction_filepath_subset = f"{config.GRAPHCAST_DATA_DIRECTORY}/perfect_model_forecasts/{prediction_initial_date}/{prediction_initial_date}_{prediction_timesteps}_subset.zarr"
perfect_predicted_data_subset = xr.open_zarr(perfect_prediction_filepath_subset).load().sel(batch=0, drop=True).drop_vars(['number', 'expver'])
datetimes = convert_ns_to_datetime(perfect_predicted_data_subset.time, init_date)
perfect_predicted_data_subset = perfect_predicted_data_subset.assign_coords({'datetime':datetimes.datetime}).swap_dims({'time':'datetime'})

logger.info("Loading initial condition data...")

initial_conditions_filepath = str(graphcast_control_predictions['initial_conditions_path'].values)
era5_data = xr.open_dataset(initial_conditions_filepath).drop_vars(['number', 'expver'])
# era5_data = xr.open_dataset(f"/glade/u/home/sressel/spencer-scratch/graphcast_input_data/{year_directory}/era5_data.nc").drop_vars(['number', 'expver'])

era5_data = era5_data.assign_coords({'datetime':era5_data.datetime.sel(batch=0, drop=True)}).swap_dims({'time':'datetime'}).sel(batch=0, drop=True)

# Load 30-year climatology for 2m temperature
era5_2m_temperature_climatology = xr.open_dataset('/glade/u/home/sressel/spencer-scratch/graphcast_input_data/climatology/06-30T00/2m_temperature_climatology_1991-2000.nc')['t2m'].mean(dim='valid_time').isel(latitude=slice(None, None, -1)).sortby('latitude').rename({'latitude':'lat', 'longitude':'lon'})
logger.info("Finished")

In [ ]:
era5_2m_temperature_climatology = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/climatology/06-30T00/2m_temperature_climatology_1979-2020.nc")['t2m'].mean(dim='valid_time').isel(latitude=slice(None, None, -1)).sortby('latitude').rename({'latitude':'lat', 'longitude':'lon'})

In [ ]:
era5_2m_temperature_climatology

In [ ]:
time_to_plot = "2021-06-30T00"

SAVE_FIG = True
PLOT_MODE = "slides"
FIG_LAYOUT = 'full'
plt.style.use('default')
set_plot_mode(PLOT_MODE)
# plt.rcParams['mathtext.fontset'] = 'dejavusans'
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
plt.rcParams['font.size'] = '12'
plt.rcParams['axes.titlesize'] = '12'
plt.rcParams['axes.labelsize'] = '12'
plt.rcParams['xtick.labelsize'] = '10'
plt.rcParams['ytick.labelsize'] = '10'
plt.rcParams['lines.linewidth'] = '1'

# 2. Define your upper threshold and out-of-bounds color
threshold = 25
color = 'yellow'  # Can be a name, hex code, or RGB tuple
# 3. Configure the colormap
cmap = colormaps.get_cmap('coolwarm')
cmap.set_over(color)            # Set color for values > threshold
cmap.set_under(color)            # Set color for values > threshold

fig = plt.figure(figsize=get_figsize(PLOT_MODE, FIG_LAYOUT))

gs = GridSpec(3, 2, figure=fig, height_ratios=[100, 100, 12.5])
gs.update(hspace=0.25)

# proj_crs = ccrs.LambertConformal(central_longitude=255, central_latitude=45, standard_parallels=(30, 60))
# proj_crs = ccrs.PlateCarree()
# proj_crs = ccrs.LambertConformal(
#     central_longitude=-122.5,
#     central_latitude=51,
#     standard_parallels=(32, 70)
# )
proj_crs = ccrs.Robinson(central_longitude=-122.5)
# proj_crs = ccrs.Mollweide()
axes = [
    fig.add_subplot(gs[0,0], projection=proj_crs),
    fig.add_subplot(gs[0,1], projection=proj_crs),
    fig.add_subplot(gs[1,0], projection=proj_crs),
    fig.add_subplot(gs[1,1], projection=proj_crs),
]
cbar_axes = [
    fig.add_subplot(gs[2, :]),
]
data_crs = ccrs.PlateCarree()

for index, data in enumerate([era5_data, graphcast_control_predictions, perfect_predicted_data_global, perfect_predicted_data_subset]):
    axes[index].set_extent([-155, -90, 32, 70], crs=data_crs)
    im = axes[index].pcolormesh(
        data['lon'],
        data['lat'],
        (data['2m_temperature'] - era5_2m_temperature_climatology).coord_funcs.sel(datetime=time_to_plot),
        transform=data_crs,
        cmap=cmap,
        # cmap='coolwarm',
        norm=mcolors.Normalize(vmin=-25, vmax=25),
        # vmin=-25, vmax=25,
        # extend='both'
    )
    cbar = fig.colorbar(im, cax=cbar_axes[0], orientation='horizontal')
    cbar.set_label("2m Air Temperature Anomalies (°C)")

    cs = axes[index].contour(
        data['lon'].sel(lon=slice(360-160, 360-80)),
        data['lat'].sel(lat=slice(30, 75)),
        (data['geopotential']/9.81).coord_funcs.sel(lat=slice(30, 75), lon=slice(360-160, 360-80), level=500, datetime=time_to_plot),
        transform=data_crs,
        levels=np.arange(4680, 5940+60, 60),
        colors='k'
        # extend='both'
    )
    axes[index].clabel(cs, inline=True, fontsize=8, colors='black')

axes[0].set_title("ERA5 verification")
axes[1].set_title("GraphCast Control")
axes[2].set_title("GraphCast global optimal")
axes[3].set_title("GraphCast regional optimal")

for axis in axes:
    axis.coastlines(resolution='50m', color='gray', linewidth=0.6)
    axis.add_feature(cfeature.BORDERS, edgecolor='gray', linewidth=0.6)
    axis.add_feature(cfeature.STATES, edgecolor='gray', linewidth=0.5)

    axis.set_aspect('equal')

    axis.add_patch(mpatches.Rectangle(xy=(230, 42), width=20, height=18,
            linestyle='--', edgecolor='magenta', facecolor='none',
            linewidth=1.5, transform=data_crs))

# fig.suptitle(
#     (
#         f"{np.datetime64(time_to_plot).astype('M8[ms]').astype('O').strftime('%HZ %d-%^b-%Y')}"
#         + f"\n{config.VARIABLE_SHORTNAMES[variable_to_plot]}" 
#         + (f"{level_to_plot}" if 'level' in era5_data[variable_to_plot].coords else "")
#     ),
#     x=0.5, y=1.01, ha="center"
# )

# plt.tight_layout()

# Save figure
output_filename = "vonich_hakim_2021_recreation_2m_temperature_anomalies"
logger.info(f"Output directory: {config.OUTPUT_DIRECTORY}/era5_control_perfect_comparisons/")
logger.info(f"Output filename: {output_filename}")
if SAVE_FIG:
    logger.info(f"Saving...")
    plt.savefig(
        f"{config.OUTPUT_DIRECTORY}/{output_filename}.svg",
        dpi=500,
        bbox_inches="tight",
    )
else:
    logger.info("Not Saving")

logger.info("Finished")
plt.show()

In [ ]:
(data['2m_temperature'] - era5_2m_temperature_climatology).sel(datetime=time_to_plot).max()

In [ ]:
era5_2m_temperature_climatology

# data['lat'].sel(lat=slice(30, 75))
# (data['geopotential']/9.81).coord_funcs.sel(lat=slice(30, 75), lon=slice(360-160, 360-80), level=500, datetime=time_to_plot)

# Calculate VPM

### Load reference VPM

In [ ]:
# Load the whitespace-delimited file
cols = ["year", "month", "day", "hour", "VPM1", "VPM2", "amplitude"]
df = pd.read_csv("/glade/u/home/sressel/spencer-scratch/vpm.1x.txt", delim_whitespace=True, names=cols)

# Build datetime64 index with hour precision (YYYY-MM-DDTHH)
time = pd.to_datetime(
    dict(year=df.year, month=df.month, day=df.day, hour=df.hour)
).values.astype("datetime64[h]")

# PC1/PC2 -> single DataArray with dims (time, mode)
reference_VPM_data = df[["VPM1", "VPM2"]].to_numpy()
reference_VPM = xr.DataArray(
    reference_VPM_data.T,
    dims=("mode", "time"),
    coords={
        "mode": [1,2],
        "time": time, 
    },
    name="VPM",
)

# Amplitude -> DataArray with dim (time,)
reference_VPM_amplitude = xr.DataArray(
    df["amplitude"].to_numpy(),
    dims="time",
    coords={"time": time},
    name="amplitude",
)

## Load data

In [ ]:
climatology_by_year = {}
years_to_load = np.arange(1990, 2020, 1)

logger.info("Loading data...")
for index, variable in enumerate(['u_component_of_wind', 'v_component_of_wind']):
    logger.info(f"-- {variable}")
    climatology_by_year[variable] = []
    for year in years_to_load:
        logger.info(f"---- {year}")
        data = xr.load_dataset(f"/glade/u/home/sressel/spencer-scratch/graphcast_input_data/6hr_climatology/{year}/{variable}.nc", decode_timedelta=True).load()[variable]
        data_retimed = data.isel(batch=0, drop=True).assign_coords({'time':data['datetime'].isel(batch=0, drop=True)}).drop_vars('datetime')
        climatology_by_year[variable].append(data_retimed.resample(time='1D').mean(dim='time'))

logger.info("Concatenating years...")
u_climatology = xr.concat([val for val in climatology_by_year['u_component_of_wind']], dim='time')
v_climatology = xr.concat([val for val in climatology_by_year['v_component_of_wind']], dim='time')
logger.info("Finished")

## Calculate EOFs

In [ ]:
save_regression_coefficients = False
save_climatological_mean = False
save_standard_deviations = False

velocity_potential_climatology = calculate_velocity_potential_era5(
    xr.merge(
    [u_climatology.sel(level=200), v_climatology]
    )
)

variables_climatology = {
    'VP200': velocity_potential_climatology.drop_vars('level'),
    'U200': u_climatology.sel(level=200).drop_vars('level'),
    'U850': u_climatology.sel(level=850).drop_vars('level'),
}
variables_std = {}
variables_processed = {}
regression_coefficients = {}
climatological_mean = {}

for variable, data in variables_climatology.items():
    logger.info(variable)

    data.name = variable

    # Meridionally average the data
    meridionally_averaged_data = data.sel(lat=slice(-15,15)).mean(dim='lat')

    # Remove the climatological mean from the data
    data_anomalies = meridionally_averaged_data.stats.standardize(dim='time', unit_variance=False)

    climatological_mean[variable] = meridionally_averaged_data.mean(dim='time')
    # climatological_mean[variable].name = f"{variable}"

    # Fit an annual cycle to the data 
    regression_coefficients[variable], t0 = fit_annual_cycle(data_anomalies, 4)
    # regression_coefficients[variable].name = f"{variable}"

    # Remove the annual cycle from the data
    data_deannualized, annual_cycle = apply_annual_cycle(
        data_anomalies, 
        regression_coefficients[variable], 
        t0, 
        4
    )

    # Remove the 120-day running mean from the data
    data_without_running_mean = data_deannualized - data_deannualized.rolling(time=120, center=False).mean().isel(time=slice(120, None))

    # Calculate the zonal-mean variance of the data and standardize the data by this value
    variables_std[variable] = np.sqrt(data_without_running_mean.var(dim='time').mean(dim='lon'))
    # variables_std[variable].name = f"{variable}"
    data_standardized = data_without_running_mean / variables_std[variable]

    variables_processed[variable] = data_standardized

if save_regression_coefficients:
    logger.info("Saving Regression Coefficients...")
    merged_regression_coefficients = xr.merge(
        [var for var in regression_coefficients.values()]
    )
    merged_regression_coefficients.attrs['t0'] = str(t0)
    merged_regression_coefficients.to_netcdf(
        f"{config.INPUT_DATA_DIRECTORY}/annual_cycle_regression_coefficients_{str(years_to_load[0])}_{str(years_to_load[-1])}.nc"
    )
    logger.info("Saved")

if save_climatological_mean:
    logger.info("Saving Climatological Mean...")
    merged_climatological_mean = xr.merge(
        [var for var in climatological_mean.values()]
    )
    merged_climatological_mean.to_netcdf(
        f"{config.INPUT_DATA_DIRECTORY}/climatological_mean_{str(years_to_load[0])}_{str(years_to_load[-1])}.nc"
    )
    logger.info("Saved")

if save_standard_deviations:
    logger.info("Saving Standard Deviations...")
    merged_standard_deviations = xr.merge(
        [var for var in variables_std.values()]
    )
    merged_standard_deviations.to_netcdf(
        f"{config.INPUT_DATA_DIRECTORY}/standard_deviations_{str(years_to_load[0])}_{str(years_to_load[-1])}.nc"
    )
    logger.info("Saved")

logger.info("Finished")

In [ ]:
save_eofs = False

multivariate_data = [
    variables_processed['VP200'],
    variables_processed['U200'],
    variables_processed['U850'],
]

model = xe.single.EOF(
    n_modes=10,
    standardize=False,   # set True if you want xeofs to z-score internally instead
    # use_coslat=True,     # applies latitude weighting — recommended for lat/lon grids
)
model.fit(multivariate_data, dim="time")

# Components and scores
components = model.components()   # returns a list of DataArrays, one per input field
scores = model.scores()           # combined PC time series (shared across variables)

explained_var = model.explained_variance_ratio()

VP200_EOF = components[0]
del VP200_EOF.attrs['solver_kwargs']
VP200_EOF.name = 'VP200'
U200_EOF = components[1]
del U200_EOF.attrs['solver_kwargs']
U200_EOF.name = 'U200'
U850_EOF = components[2]
del U850_EOF.attrs['solver_kwargs']
U850_EOF.name = 'U850'

merged_EOFs = xr.merge(
    [VP200_EOF.sel(mode=[1,2]), U200_EOF.sel(mode=[1,2]), U850_EOF.sel(mode=[1,2])]
)
if save_eofs:
    logger.info("Saving EOF structures...")
    merged_EOFs.to_netcdf(f"{config.INPUT_DATA_DIRECTORY}/EOFs_for_projection_{str(years_to_load[0])}_{str(years_to_load[-1])}.nc")
    logger.info("Saved")

PCs = scores / scores.sel(mode=1).std()
VPM = np.sqrt(PCs.sel(mode=1)**2 + PCs.sel(mode=2)**2)

plot_eofs = True
if plot_eofs:
    [fig, ax] = plt.subplots(3, 1, figsize=(9,16))

    VP200_EOF.sel(mode=1).plot(ax=ax[0], color='k')
    U200_EOF.sel(mode=1).plot(ax=ax[0], color='k', ls=':')
    U850_EOF.sel(mode=1).plot(ax=ax[0], color='k', ls='--')
    ax[0].axhline(y=0, color='k', lw=1)

    VP200_EOF.sel(mode=2).plot(ax=ax[1], color='k')
    U200_EOF.sel(mode=2).plot(ax=ax[1], color='k', ls=':')
    U850_EOF.sel(mode=2).plot(ax=ax[1], color='k', ls='--')
    ax[1].axhline(y=0, color='k', lw=1)

    VP200_EOF.sel(mode=3).plot(ax=ax[2], color='k')
    U200_EOF.sel(mode=3).plot(ax=ax[2], color='k', ls=':')
    U850_EOF.sel(mode=3).plot(ax=ax[2], color='k', ls='--')
    ax[2].axhline(y=0, color='k', lw=1)

    plt.show()

In [ ]:
PC_to_save =scores.sel(mode=1).std()
del PC_to_save.attrs['solver_kwargs']
PC_to_save.to_netcdf("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/PC_standard_deviation_1990_2019.nc")

In [ ]:
PC_to_save = scores.sel(mode=1, drop=True).std()
del PC_to_save.attrs['solver_kwargs']
PC_to_save.to_netcdf("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/PC_standard_deviation_1990_2019.nc")

In [ ]:
start_date = "2011-11-01"
end_date = "2011-12-31"

plt.plot(
    PCs.sel(mode=1).sel(time=slice(np.datetime64(start_date), np.datetime64(end_date))),
    PCs.sel(mode=2).sel(time=slice(np.datetime64(start_date), np.datetime64(end_date)))
)
plt.plot(
    reference_VPM.sel(mode=1).sel(time=slice(np.datetime64(start_date), np.datetime64(end_date))),
    reference_VPM.sel(mode=2).sel(time=slice(np.datetime64(start_date), np.datetime64(end_date)))
)

In [ ]:
concatenated_EOFs = xr.concat(
    [VP200_EOF, U200_EOF, U850_EOF],
    dim='lon'
)
concatenated_EOFs

In [ ]:
annual_cycle_regression_coefficients = xr.merge(
    [regression_coefficients[var] for var in regression_coefficients.keys()]
)

In [ ]:
regression_coeffs = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/annual_cycle_regression_coefficients_1990_2009.nc")

climatological_annual_cycle = reconstruct_annual_cycle(
    data_anomalies.time,
    regression_coeffs['VP200'],
    np.datetime64(regression_coeffs.attrs['t0'])
)

In [ ]:
from auxiliary_functions.calculate_velocity_potential import calculate_velocity_potential_era5
from auxiliary_functions.mjo_mean_state_diagnostics import fit_annual_cycle, apply_annual_cycle

velocity_potential_climatology = calculate_velocity_potential_era5(
    xr.merge(
    [u_climatology.sel(level=200), v_climatology]
    )
)

variables_climatology = {
    'VP200': velocity_potential_climatology,
    'U200': u_climatology.sel(level=200),
    'U850': u_climatology.sel(level=850),
}
variables_std = {}
variables_processed = {}
regression_coefficients = {}
climatological_mean_and_annual_cycle = {}

for variable, data in variables_climatology.items():
    logger.info(variable)
        
    meridionally_averaged_data = data.sel(lat=slice(-15,15)).mean(dim='lat')

    data_anomalies = meridionally_averaged_data.stats.standardize(dim='time', unit_variance=False)

    regression_coefficients[variable], t0 = fit_annual_cycle(data_anomalies, 4)
    data_deannualized, annual_cycle = apply_annual_cycle(
        data_anomalies, 
        regression_coefficients[variable], 
        t0, 
        4
    )

    data_without_running_mean = data_deannualized - data_deannualized.rolling(time=120, center=False).mean().isel(time=slice(120, None))

    variables_std[variable] = np.sqrt(data_without_running_mean.var(dim='time').mean(dim='lon'))
    data_standardized = data_without_running_mean / variables_std[variable]

    climatological_mean_and_annual_cycle[variable] = annual_cycle + meridionally_averaged_data.mean(dim='time')
    variables_processed[variable] = data_standardized

### Load data to be projected

In [ ]:
U_proj = xr.load_dataset(f"/glade/u/home/sressel/spencer-scratch/graphcast_input_data/6hr_climatology/2018/u_component_of_wind.nc").load()['u_component_of_wind']
U_proj_retimed = U_proj.isel(batch=0, drop=True).assign_coords({'time':U_proj['datetime'].isel(batch=0, drop=True)})
U_proj_daily = U_proj_retimed.resample(time='1D').mean(dim='time')

V_proj = xr.load_dataset(f"/glade/u/home/sressel/spencer-scratch/graphcast_input_data/6hr_climatology/2018/v_component_of_wind.nc", decode_timedelta=True).load()['v_component_of_wind']
V_proj_retimed = V_proj.isel(batch=0, drop=True).assign_coords({'time':V_proj['datetime'].isel(batch=0, drop=True)}).drop_vars('datetime')
V_proj_daily = V_proj_retimed.resample(time='1D').mean(dim='time')

VP200_proj = calculate_velocity_potential_era5(
    xr.merge(
    [U_proj_daily.sel(level=200), V_proj_daily]
    )
)

variables_projection_climatology = {
    # 'VP200': VP200_proj,
    # 'U200': U_proj_daily.sel(level=200),
    'U850': U_proj_daily.sel(level=850),
}
variables_projection_processed = {}

for variable, data in variables_projection_climatology.items():
    logger.info(variable)
        
    meridionally_averaged_data = data.sel(lat=slice(-15,15)).mean(dim='lat')

    data_anomalies = meridionally_averaged_data.stats.standardize(dim='time', unit_variance=False)

    data_deannualized, annual_cycle = apply_annual_cycle(
        data_anomalies, 
        regression_coefficients[variable], 
        t0, 
        4
    )

    data_without_running_mean = data_deannualized - data_deannualized.rolling(time=120, center=False).mean().isel(time=slice(120, None))

#     data_standardized = data_without_running_mean / variables_std[variable]

#     variables_projection_processed[variable] = data_standardized


In [ ]:
data_anomalies

### Calculate PCs of projection data

In [ ]:
concatenated_projection_data = xr.concat(
    [variables_projection_processed['VP200'],
    variables_projection_processed['U200'],
    variables_projection_processed['U850']],
    dim='lon'
)


concatenated_EOFs = xr.concat(
    [
        VP200_EOF.sel(mode=[1,2]),
        U200_EOF.sel(mode=[1,2]),
        U850_EOF.sel(mode=[1,2]),
    ],
    dim='lon'
)

projected_PCs = xr.dot(
    concatenated_EOFs,
    concatenated_projection_data,
    dim='lon'
)

projected_VPM1 = projected_PCs.sel(mode=1) / projected_PCs.sel(mode=1).std()
projected_VPM2 = projected_PCs.sel(mode=2) / projected_PCs.sel(mode=1).std()

## Load Hera's EOFs

In [ ]:
# hera_eofs = hera_eofs.rename({'u200':'U200', 'u850':'U850', 'vp200':'VP200'})
hera_eofs.attrs['std'] = [5.576273e+00, 1.853310e+00, 4.718959e+06]
hera_eofs.to_netcdf("/glade/u/home/sressel/spencer-scratch/VPM_EOFs_ERA-I_1990-2009.nc")

In [ ]:
hera_eofs = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/eofev_VPM_ERA-I_1990-2009_.nc").isel(lev=0, lat=0, drop=True).rename({'time':'mode'}).assign_coords({'mode':[1,2,3,4,5]}).interp(lon=np.arange(0.0, 360.0, 1.0)).interpolate_na(dim='lon')
hera_eofs['u200'][:, -1] = (hera_eofs['u200'][:, -2] + hera_eofs['u200'][:, 0])/2
hera_eofs['u850'][:, -1] = (hera_eofs['u850'][:, -2] + hera_eofs['u850'][:, 0])/2
hera_eofs['vp200'][:, -1] = (hera_eofs['vp200'][:, -2] + hera_eofs['vp200'][:, 0])/2

hera_stds = xr.DataArray(
    data=np.array([5.576273, 1.853310, 4718959.]),
    dims='variable',
    coords={'variable':['U200', 'U850', 'VP200']}
)


concat_data = np.concat(
    [
        rescaled_U200.isel(batch=0, drop=True).values[10:],
        rescaled_U850.isel(batch=0, drop=True).values[10:],
        rescaled_VP200.isel(batch=0, drop=True).values[10:],    
    ],
    axis=1
)

concat_eofs = np.concat(
    [
        hera_eofs['u200'].sel(mode=[1,2]).values,
        hera_eofs['u850'].sel(mode=[1,2]).values,
        hera_eofs['vp200'].sel(mode=[1,2]).values,
    ],
    axis=1
)

In [ ]:
PCs = np.einsum('ij,kj->ik', concat_eofs, concat_data)
VPM1 = PCs[0]/np.std(PCs[0])
VPM2 = PCs[1]/np.std(PCs[0])
loss = VPM1 + VPM2
loss

In [ ]:

# u850_norm = U850.sel(lat=slice(-15,15)).mean(dim='lat').isel(batch=0, drop=True).stats.standardize(zero_mean=False, unit_variance=True)
# u200_norm = U200.sel(lat=slice(-15,15)).mean(dim='lat').isel(batch=0, drop=True).stats.standardize(zero_mean=False, unit_variance=True)
# vp200_norm = VP200.sel(lat=slice(-15,15)).mean(dim='lat').isel(batch=0, drop=True).stats.standardize(zero_mean=False, unit_variance=True)

# u850_norm = U850.sel(lat=slice(-15,15)).mean(dim='lat').stats.standardize(zero_mean=False, unit_variance=True)
# u200_norm = U200.sel(lat=slice(-15,15)).mean(dim='lat').stats.standardize(zero_mean=False, unit_variance=True)
# vp200_norm = VP200.sel(lat=slice(-15,15)).mean(dim='lat').stats.standardize(zero_mean=False, unit_variance=True)

# multivariate_data = np.concatenate(
#     [
#         vp200_norm,
#         u850_norm,
#         u200_norm,
#     ], 
#     axis=1
# )

# multivariate_data = [
#     vp200_norm,
#     u850_norm,
#     u200_norm,
# ]

# model = xe.single.EOF(
#     n_modes=10,
#     standardize=False,   # set True if you want xeofs to z-score internally instead
#     # use_coslat=True,     # applies latitude weighting — recommended for lat/lon grids
# )
# model.fit(multivariate_data, dim="time")

# # Components and scores
# components = model.components()   # returns a list of DataArrays, one per input field
# scores = model.scores()           # combined PC time series (shared across variables)

# explained_var = model.explained_variance_ratio()
# logger.info("Finished")

# U, S, VT = np.linalg.svd(multivariate_data.T, full_matrices=False)
# EOF = U.T 
# PC = np.dot(np.diag(S), VT) 


In [ ]:
hera_eofs = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/eofev_VPM_ERA-I_1990-2009_.nc").isel(lev=0, lat=0, drop=True).rename({'time':'mode'}).assign_coords({'mode':[1,2,3,4,5]}).interp(lon=np.arange(0.0, 366.0, 1.0))
hera_eofs['u200'].sel(mode=1).plot(x='lon')
hera_eofs['u850'].sel(mode=1).plot(x='lon')
hera_eofs['vp200'].sel(mode=1).plot(x='lon')

In [ ]:
hera_eofs_interpolated = hera_eofs.interp(lon=np.arange(0.0, 366.0, 1.0))
hera_eofs_interpolated['u200'].sel(mode=1).plot()
hera_eofs_interpolated['u850'].sel(mode=1).plot()
hera_eofs_interpolated['vp200'].sel(mode=1).plot()

In [ ]:
np.sqrt((xr.concat(
    [
        hera_eofs['u200'].sel(mode=5),
        hera_eofs['u850'].sel(mode=5),
        hera_eofs['vp200'].sel(mode=5)
    ],
    dim='lon'
)**2).sum())

In [ ]:
from auxiliary_functions.mjo_mean_state_diagnostics import remove_annual_cycle
import numpy as np

VP200_merid = VP200.sel(lat=slice(-15,15)).mean(dim='lat')
U200_merid = U200.sel(lat=slice(-15,15)).mean(dim='lat')
U850_merid = U850.sel(lat=slice(-15,15)).mean(dim='lat')

VP200_deano, _ = remove_annual_cycle(VP200_merid, nharmonics=4)
U200_deano, _ = remove_annual_cycle(U200_merid, nharmonics=4)
U850_deano, _ = remove_annual_cycle(U850_merid, nharmonics=4)

VP200_highpass = VP200_deano - VP200_deano.rolling(time=120, center=False).mean()
U200_highpass = U200_deano - U200_deano.rolling(time=120, center=False).mean()
U850_highpass = U850_deano - U850_deano.rolling(time=120, center=False).mean()

VP200_highpass_norm = np.sqrt(VP200_highpass.var(dim='time').mean(dim='lon'))
U200_highpass_norm = np.sqrt(U200_highpass.var(dim='time').mean(dim='lon'))
U850_highpass_norm = np.sqrt(U850_highpass.var(dim='time').mean(dim='lon'))

VP200_highpass_std = VP200_highpass / VP200_highpass_norm
U200_highpass_std = U200_highpass / U200_highpass_norm
U850_highpass_std = U850_highpass / U850_highpass_norm

multivariate_data = [
    VP200_highpass_std,
    U200_highpass_std,
    U850_highpass_std,
]

model = xe.single.EOF(
    n_modes=10,
    standardize=False,   # set True if you want xeofs to z-score internally instead
    # use_coslat=True,     # applies latitude weighting — recommended for lat/lon grids
)
model.fit(multivariate_data, dim="time")

# Components and scores
components = model.components()   # returns a list of DataArrays, one per input field
scores = model.scores()           # combined PC time series (shared across variables)


In [ ]:
VP200_new_highpass
U200_new_highpass
U850_new_highpass

VP200_new_std = VP200_new_highpass / vp200_norm
U200_new_std  = U200_new_highpass  / u200_norm
U850_new_std  = U850_new_highpass  / u850_norm

new_multivariate = [VP200_new_std, U200_new_std, U850_new_std]
new_scores = model.transform(new_multivariate)

In [ ]:
import matplotlib.pyplot as plt 

VP200_EOF = components[0]
U200_EOF = components[1]
U850_EOF = components[2]

[fig, ax] = plt.subplots(3, 1, figsize=(9,16))

VP200_EOF.sel(mode=1).plot(ax=ax[0], color='k')
U200_EOF.sel(mode=1).plot(ax=ax[0], color='k', ls=':')
U850_EOF.sel(mode=1).plot(ax=ax[0], color='k', ls='--')
ax[0].axhline(y=0, color='k', lw=1)

VP200_EOF.sel(mode=2).plot(ax=ax[1], color='k')
U200_EOF.sel(mode=2).plot(ax=ax[1], color='k', ls=':')
U850_EOF.sel(mode=2).plot(ax=ax[1], color='k', ls='--')
ax[1].axhline(y=0, color='k', lw=1)

VP200_EOF.sel(mode=3).plot(ax=ax[2], color='k')
U200_EOF.sel(mode=3).plot(ax=ax[2], color='k', ls=':')
U850_EOF.sel(mode=3).plot(ax=ax[2], color='k', ls='--')
ax[2].axhline(y=0, color='k', lw=1)

plt.show()

In [ ]:
VP200_new = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/ECMWF/ERA5/daily_data/daily_25_degree_meridional_wind_1980_2018.nc")['v'].sel(plev=200, drop=True).sel(time=slice('2010-01-01T00:00:00.000000000', '2010-01-31T00:00:00.000000000'))

U200_new = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/ECMWF/ERA5/daily_data/daily_25_degree_zonal_wind_1980_2018.nc")['u'].sel(plev=200, drop=True).sel(time=slice('2010-01-01T00:00:00.000000000', '2010-01-31T00:00:00.000000000'))

U850_new = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/ECMWF/ERA5/daily_data/daily_25_degree_zonal_wind_1980_2018.nc")['u'].sel(plev=850, drop=True).sel(time=slice('2010-01-01T00:00:00.000000000', '2010-01-31T00:00:00.000000000'))

VP200_merid_new = VP200_new.sel(lat=slice(-15,15)).mean(dim='lat')
U200_merid_new = U200_new.sel(lat=slice(-15,15)).mean(dim='lat')
U850_merid_new = U850_new.sel(lat=slice(-15,15)).mean(dim='lat')


VP200_deano_new, _ = apply_annual_cycle(VP200_merid_new, VP200_coeffs, VP200_t0, nharmonics=4)

In [ ]:
# Configure plot
# import seaborn as sns
plt.style.use('default')
[fig, ax] = plt.subplots(1, 2, figsize=(12,6))
# plt.xlabel("RMM1")
# plt.ylabel("RMM2")

# Plot index points
# colormap = sns.color_palette("viridis", as_cmap=True)

# start_time = '1992-02-12' 
# end_time = '1992-05-12'

# start_time = '1992-08-14' 
# end_time = '1992-11-12'

# start_time = np.datetime64("1975-04-01T00:00:00.000000000")
# end_time = np.datetime64("1975-04-30T00:00:00.000000000")

# start_time = '1975-02-24T00:00:00.000000000'
# end_time = '1975-04-11T00:00:00.000000000'

# start_time="2020-12-29T00:00:00.000000000"
# end_time="2021-02-27T00:00:00.000000000"

start_time = 0
end_time = 2138400000000000

# start_time = primary_event_start_times[-1].values
# end_time = primary_event_end_times[-1].values

# for index, (start_time, end_time) in enumerate(zip(('1992-02-12', '1992-08-14'), ('1992-05-12', '1992-11-12'))):


# for time in principle_components.time.sel(time=slice(start_time, end_time)):
ax[index].plot(
    principle_components.sel(mode=1).isel(time=slice(start_time, end_time)),
    principle_components.sel(mode=2).isel(time=slice(start_time, end_time)),
    color='k',
    linestyle='-',
    marker=".",
    ms=8,
)

# Add phase regions overlay
circle1 = plt.Circle((0, 0), 0.3, color="#bcbcbc", fill=False, lw=1, zorder=10)
circle2 = plt.Circle((0, 0), 0.4, color="k", fill=False, lw=1.5, zorder=10)
circle3 = plt.Circle((0, 0), 0.5, color="#bcbcbc", fill=False, lw=1, zorder=10)
ax[index].add_patch(circle1)
ax[index].add_patch(circle2)
ax[index].add_patch(circle3)

ax[index].axhline(y=0, color="k", lw=1, ls="-")
ax[index].axvline(x=0, color="k", lw=1, ls="-")

# Add lines to differentiate the phases
ax[index].plot(
    [0.4*np.cos(np.deg2rad(45)), 10*np.cos(np.deg2rad(45))],
    [0.4*np.sin(np.deg2rad(45)), 10*np.sin(np.deg2rad(45))],
    color="k",
    lw=1,
    ls="-"
)
ax[index].plot(
    [0.4*np.cos(np.deg2rad(45)), 10*np.cos(np.deg2rad(45))],
    [-0.4*np.sin(np.deg2rad(45)), -10*np.sin(np.deg2rad(45))],
    color="k",
    lw=1,
    ls="-"
)
ax[index].plot(
    [-0.4*np.cos(np.deg2rad(45)), -10*np.cos(np.deg2rad(45))],
    [-0.4*np.sin(np.deg2rad(45)), -10*np.sin(np.deg2rad(45))],
    color="k",
    lw=1,
    ls="-"
)
ax[index].plot(
    [-0.4*np.cos(np.deg2rad(45)), -10*np.cos(np.deg2rad(45))],
    [0.4*np.sin(np.deg2rad(45)), 10*np.sin(np.deg2rad(45))],
    color="k",
    lw=1,
    ls="-"
)

ax[index].set_xlim(-3,3)
ax[index].set_ylim(-3,3)

# # Add phase labels
ax[index].text(
    2.9, 0.1,
    f'Category A',
    horizontalalignment='right',
    verticalalignment='center',
    fontsize=12
)
ax[index].text(
    0, 2.8,
    f'Category B',
    horizontalalignment='center',
    verticalalignment='center',
    fontsize=12
)
ax[index].text(
    -2.9, .1,
    f'Category C',
    horizontalalignment='left',
    verticalalignment='center',
    fontsize=12
)
ax[index].text(
    0, -2.8,
    f'Category D',
    horizontalalignment='center',
    verticalalignment='center',
    fontsize=12
)

ax[index].spines['left'].set_position('zero')
ax[index].spines['bottom'].set_position('zero')

# Hide the top and right spines
ax[index].spines['right'].set_color('none')
ax[index].spines['top'].set_color('none')

# Ensure tick marks follow the spines to the center
ax[index].xaxis.set_ticks_position('bottom')
ax[index].yaxis.set_ticks_position('left')

ax[index].set_aspect("equal")
plt.tight_layout()

In [ ]:
# vars_to_keep = ['u_component_of_wind', 'v_component_of_wind']
predictions = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/2020-09_2021-02/era5_data.nc")
# predictions = predictions.drop_vars([var for var in predictions.data_vars if var not in vars_to_keep])
predictions = predictions.drop_vars(['number', 'expver', 'datetime'])
targets = xr.zeros_like(predictions)
for var in predictions.data_vars:
    logger.info(var)
    if var not in ['geopotential_at_surface', 'land_sea_mask']:
        targets[var][:] =  (
            predictions[var].values
            + np.random.normal(predictions[var].mean().values, predictions[var].std().values, size=predictions[var].shape)
        )
    else:
        targets[var][:] =  predictions[var].values

logger.info("Finished")

In [ ]:
init_date = "2021-01-01"

from auxiliary_functions import calculate_velocity_potential_graphcast

meridional_mean_climatological_data = xr.open_dataset(
    "/glade/u/home/sressel/spencer-scratch/graphcast_input_data/daily_climatology/climatology_for_graphcast.nc"
)

def recalculate_timestamps(data, init_date):
    init_time = data.where(data.datetime == np.datetime64(init_date), drop=True).time
    new_times = data.time.values - init_time.values
    retimed_data = data.assign_coords({'time':new_times})
    return retimed_data

def remove_rolling_mean(data, init_date):
    reference_date = np.datetime64(init_date)  
    decoded_time = reference_date + data["time"].values.astype("timedelta64[ns]")
    data_with_date = data.assign_coords(time=decoded_time)

    time_resampled_data = data_with_date.resample({'time':'1D'}).mean(dim='time')
    data_anomalies = time_resampled_data - time_resampled_data.rolling(
        time=120, center=False
    ).mean(dim='time')
    return data_anomalies

# Load in the climatological data for a given init_date
meridional_mean_climatology = xr.open_dataset(
    "/glade/u/home/sressel/spencer-scratch/graphcast_input_data/daily_climatology/climatology_for_graphcast.nc"
)

rolling_mean_climatology = xr.open_dataset(
    "/glade/u/home/sressel/spencer-scratch/graphcast_input_data/2020-05_2021-02/era5_data.nc"
)

# Rewrite the timestamps on the climatological data to match the predictions
rolling_mean_climatology = recalculate_timestamps(rolling_mean_climatology, init_date)

# Append the graphcast predictions to the climatological data
predictions_appended = xr.concat(
    [rolling_mean_climatology, predictions],
    dim='time'
)

# Append the graphcast predictions to the climatological data
targets_appended = xr.concat(
    [rolling_mean_climatology, targets],
    dim='time'
)

# Extract U200 and U850 from meridionally averaged data
U200_predictions = predictions_appended['u_component_of_wind'].sel(level=200)
U200_targets = targets_appended['u_component_of_wind'].sel(level=200)

U850_predictions = predictions_appended['u_component_of_wind'].sel(level=850)
U850_targets = targets_appended['u_component_of_wind'].sel(level=850)

# Calculate velocity potential for the predictions and targets
VP200_predictions = calculate_velocity_potential_graphcast(
    predictions_appended
)
VP200_targets = calculate_velocity_potential_graphcast(
    targets_appended
)

def process_data_for_EOFs(data):
    meridioanlly_averaged_data = data.sel(lat=slice(-15,15)).mean(dim='lat')

    



# Meridionally average targets and predictions
meridionally_averaged_U200_predictions = U200_predictions.sel(
    lat=slice(-15,15)
).mean(dim='lat')
meridionally_averaged_U850_predictions = U850_predictions.sel(
    lat=slice(-15,15)
).mean(dim='lat')
meridionally_averaged_VP200_predictions = VP200_predictions.sel(
    lat=slice(-15,15)
).mean(dim='lat')

meridionally_averaged_U200_targets = U200_targets.sel(
    lat=slice(-15,15)
).mean(dim='lat')
meridionally_averaged_U850_targets = U850_targets.sel(
    lat=slice(-15,15)
).mean(dim='lat')
meridionally_averaged_VP200_targets = VP200_targets.sel(
    lat=slice(-15,15)
).mean(dim='lat')

climatology = xr.open_dataset("")

# Remove 120-rolling mean climatology from the variables
U200_prediction_anomalies = remove_rolling_mean(
    meridionally_averaged_U200_predictions,
    init_date
)
U200_target_anomalies = remove_rolling_mean(
    meridionally_averaged_U200_targets,
    init_date
)
U850_prediction_anomalies = remove_rolling_mean(
    meridionally_averaged_U850_predictions,
    init_date
)
U850_target_anomalies = remove_rolling_mean(
    meridionally_averaged_U850_targets,
    init_date
)
VP200_prediction_anomalies = remove_rolling_mean(
    meridionally_averaged_VP200_predictions,
    init_date
)
VP200_target_anomalies = remove_rolling_mean(
    meridionally_averaged_VP200_targets,
    init_date
)

VPM_EOF_filepath = "/glade/u/home/sressel/spencer-scratch/VPM_EOFs_ERA-I_1990-2009.nc"
EOFs = xr.open_dataset(VPM_EOF_filepath).isel(
    lev=0, lat=0, drop=True
).rename(
    {'time':'mode'}
).assign_coords(
    {'mode':[1,2,3,4,5]}
).interp(
    lon=np.arange(0.0, 360.0, 1.0)
).interpolate_na(dim='lon')
EOFs['U200'][:, -1] = (EOFs['U200'][:, -2] + EOFs['U200'][:, 0])/2
EOFs['U850'][:, -1] = (EOFs['U850'][:, -2] + EOFs['U850'][:, 0])/2
EOFs['VP200'][:, -1] = (EOFs['VP200'][:, -2] + EOFs['VP200'][:, 0])/2

concatenated_eofs = np.concat(
    [
        EOFs['U200'].sel(mode=[1,2]).values,
        EOFs['U850'].sel(mode=[1,2]).values,
        EOFs['VP200'].sel(mode=[1,2]).values,
    ],
    axis=1
)

standard_deviations = EOFs.attrs['std']

U200_prediction_anomalies_rescaled = (
    U200_prediction_anomalies / standard_deviations[0]
)
U200_target_anomalies_rescaled = (
    U200_target_anomalies / standard_deviations[0]
)
U850_prediction_anomalies_rescaled = (
    U850_prediction_anomalies / standard_deviations[1]
)
U850_target_anomalies_rescaled = (
    U850_target_anomalies / standard_deviations[1]
)
VP200_prediction_anomalies_rescaled = (
    VP200_prediction_anomalies / standard_deviations[2]
)
VP200_target_anomalies_rescaled = (
    VP200_target_anomalies / standard_deviations[2]
)

predictions_concatenated = np.concat(
    [
        U200_prediction_anomalies_rescaled.isel(batch=0, drop=True).values,
        U850_prediction_anomalies_rescaled.isel(batch=0, drop=True).values,
        VP200_prediction_anomalies_rescaled.isel(batch=0, drop=True).values,    
    ],
    axis=1
)

targets_concatenated = np.concat(
    [
        U200_target_anomalies_rescaled.isel(batch=0, drop=True).values,
        U850_target_anomalies_rescaled.isel(batch=0, drop=True).values,
        VP200_target_anomalies_rescaled.isel(batch=0, drop=True).values,    
    ],
    axis=1
)

difference_PCs = np.einsum(
    'ij,kj->ik', 
    concatenated_eofs, 
    (predictions_concatenated - targets_concatenated)
)
difference_VPM1 = difference_PCs[0]/np.std(difference_PCs[0])
difference_VPM2 = difference_PCs[1]/np.std(difference_PCs[0])

losses = np.sqrt(difference_VPM1**2 + difference_VPM2**2)

# Misc

In [ ]:
import re
from pathlib import Path
from datetime import datetime

def find_era5_data(target_date: str, parent_dir: str, filename: str = "era5_data.nc") -> Path:
    """
    Locate the ERA5 data file whose containing folder (named YYYY-MM_YYYY-MM)
    spans the given target date.

    Parameters
    ----------
    target_date : str
        Date in 'YYYY-MM-DD' format (time component, if present, is ignored).
    parent_dir : str
        Directory containing the YYYY-MM_YYYY-MM subfolders.
    filename : str
        Name of the file to locate within the matching folder (default: 'era5_data.nc').

    Returns
    -------
    Path
        Full path to the matching data file.

    Raises
    ------
    ValueError
        If target_date can't be parsed, no folder matches, multiple folders
        match, or the expected file doesn't exist in the matched folder.
    """
    # Parse just the date portion (drop any 'T...' time component if present)
    date_str = target_date.split("T")[0]
    try:
        target = datetime.strptime(date_str, "%Y-%m-%d")
    except ValueError as e:
        raise ValueError(f"Invalid date '{target_date}': {e}")

    target_ym = target.year * 100 + target.month  # e.g. 202010

    pattern = re.compile(r"^(\d{4})-(\d{2})_(\d{4})-(\d{2})$")
    parent = Path(parent_dir)

    matches = []
    for entry in parent.iterdir():
        if not entry.is_dir():
            continue
        m = pattern.match(entry.name)
        if not m:
            continue

        start_ym = int(m.group(1)) * 100 + int(m.group(2))
        end_ym = int(m.group(3)) * 100 + int(m.group(4))

        if start_ym <= target_ym <= end_ym:
            matches.append(entry)

    if not matches:
        raise ValueError(f"No folder in {parent_dir} spans {date_str}")
    if len(matches) > 1:
        raise ValueError(
            f"Multiple folders span {date_str}: {[str(m) for m in matches]}. "
            "Resolve the overlap before proceeding."
        )

    data_file = matches[0] / filename
    if not data_file.is_file():
        raise ValueError(f"Matched folder {matches[0]}, but {data_file} does not exist")

    return data_file

In [ ]:
mjo_start_dates = [
    "1975-02-24", "1975-11-23", "1980-09-26", "1981-07-11", "1981-11-13", "1984-09-18", "1985-03-31", "1986-02-25", "1986-10-18", "1987-11-28", "1988-10-31", "1989-06-25", "1990-07-15", "1990-09-12", "1992-09-18", "1992-11-17", "1993-11-21", "1994-04-22", "1994-06-23", "1997-06-21", "1997-08-27", "1998-10-19", "1999-08-14", "2001-04-08", "2001-10-19", "2003-11-19", "2004-07-19", "2004-09-16", "2006-03-12", "2007-07-12", "2008-08-15", "2008-11-07", "2012-05-20", "2016-10-15", "2017-11-13", "2020-09-24", "2020-12-28", "2022-03-06"
]

In [ ]:
subdirectories = sorted(glob.glob("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/*"))
for directory in subdirectories:
    has_era5 = (True if glob.glob(f"{directory}/era5_data.nc") else False)
    print(directory.split("/")[-1], has_era5)

In [ ]:
xr.open_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/1992-08_1992-11/era5_data.nc").datetime.values